In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py
import scipy.stats as sts
import qp

In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 30

plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)   # fontsize of the figure title
plt.rc('xtick', direction = 'in')
plt.rc('ytick', direction = 'in')
#cmap = sns.color_palette('colorblind')
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
bands = ['mag_u_lsst', 
         'mag_g_lsst',
         'mag_r_lsst',
         'mag_i_lsst',
         'mag_z_lsst',
         'mag_y_lsst'
         ]

band_names = ['u', 
              'g',
              'r',
              'i',
              'z',
              'y'
              ]

specs = ['BOSS',
        'GAMA',
        'HSC',  
        'zCOSMOS',
        'DEEP2', 
        'VVDSf02',
        ]

invz = ['invz=0.1', 
        'invz=0.3',
        'invz=0.6', 
        'invz=1.0', 
        'invz=1.4', 
        'LSSTerr_only'
        ]

invz_names = ['$z_0 = 0.1$', 
              '$z_0 = 0.3$',
                '$z_0 = 0.6$',
                '$z_0 = 1.0$',
                '$z_0 = 1.4$',
                'LSST Error Model Only'
                 ]

deg_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', 
          'invz=0.1',
          'invz=0.3', 
          'invz=0.6', 
          'invz=1.0', 
          'invz=1.4', 
          'LSSTerr_only', 
          #'LSSTerr_null'
          ]

In [ ]:
def removeNans(df, cols):
    ls = []
    for col in cols: 
        ls +=  df.index[np.isnan(df[col]) ==True].tolist()

    drops = list(set(ls))
    new_df = df.drop(drops)

    return new_df 

In [ ]:
def gethdf5(est, deg, gal):

    if est != 'PZFlow':
        file = h5py.File(est+'/'+deg+'/output_estimate_'+est+'.hdf5')
    else: 
        file = h5py.File(est+'/'+deg+'/output_estimate_pzflow.hdf5')

    if est == 'CMNN' or est=='GPz':
        grid = np.linspace(0, 3, 301)
        return grid, (file['data']['loc'][gal], file['data']['scale'][gal])
    else: 
        return file['meta']['xvals'][0], file['data']['yvals'][gal]

In [ ]:
def getEns(est, deg): 
    if est == 'GPz_GL':
        path ='../paper_data/GPz_GL/'+deg+'/output_estimate_GPz.hdf5'
    elif est != 'PZFlow': 
        path =est+'/'+deg+'/output_estimate_'+est+'.hdf5'
    else: 
        path = est+'/'+deg+'/output_estimate_pzflow.hdf5'
    return qp.read(path)

# Visualizing Training Data

In [ ]:
def getTrainFile(deg):
    return (pd.read_parquet('../paper_data/CMNN/'+deg+'/output_quantity_cut_1.pq'))

LSSTerr_only = pd.read_parquet('CMNN/LSSTerr_only/output_quantity_cut_2.pq')

In [ ]:
fig, axes = plt.subplots(nrows = 6, ncols = 6, figsize = (80, 40))

for i in range(0, 6):
    for j in range(0, 6):
        file = LSSTerr_only
        axes[i][j].scatter(file['redshift'], file[bands[i]], s=0.5)

for k in range(0, 6):
    deg = specs[k]
    file = getTrainFile(deg)
    for l in range(0, 6):
        axes[l][k].scatter(file['redshift'], file[bands[l]], s=0.5)


mpl.rcParams.update({'font.size':22})

for i in range(0, 6):
    axes[0][i].set_title(specs[i])

for i in range(0, 6):
    axes[i][0].set_ylabel(band_names[i])


mpl.rcParams.update({'font.size':30})

fig.supxlabel('Redshift', size = 'x-large')
fig.supylabel('Magnitude', size = 'x-large')
fig.suptitle('Spectroscopic Survey Degradation', size = 'xx-large')

In [ ]:
file = getTrainFile('zCOSMOS')
x = file['redshift'].tolist()
y = file[bands[5]].tolist()
plt.scatter(x, y, s=0.5)

tuples = []
for i in range(len(x)):
    tuple = (x[i], y[i])
    tuples.append(tuple)


zstart = 0
zstop = 3
magstart = 18
magstop = 32

datagrid = []

z = 0
while z <= 3:

    inzlims = []
    for tuple in tuples: 
        if z <= tuple[0] <= z + 0.1:
            inzlims.append(tuple)
    
    mags = []
    mag = 16
    while mag <= 26:
        gals = 0
        for tuple in inzlims: 
            if mag <= tuple[1] <= mag + 1:
                gals += 1
        mags.append(gals)
        mag += 0.5
    
    datagrid.append(mags)
    z += 0.1


print(datagrid)

In [ ]:
df1 = pd.DataFrame(datagrid)
df2 = pd.DataFrame(datagrid).transpose()

df1


In [ ]:
df2.to_parquet('../paper_data/zCOSMOS_y_band_grid.pq')
df2

In [ ]:
testset = pd.read_parquet('./test_set_y_band_grid.pq')
BOSS = pd.read_parquet('./BOSS_y_band_grid.pq')
DEEP2 = pd.read_parquet('./DEEP2_y_band_grid.pq')
GAMA = pd.read_parquet('./GAMA_y_band_grid.pq')
HSC = pd.read_parquet('./HSC_y_band_grid.pq')
VVDSf02 = pd.read_parquet('./VVDSf02_y_band_grid.pq')
zCOSMOS = pd.read_parquet('./zCOSMOS_y_band_grid.pq')

In [ ]:
testset

In [ ]:
testset_cells = 0

cols = 29
rows = 20

for i in range(0, cols):
    for j in range(0, rows): 
        if testset[i][j]>=1:
            testset_cells += 1


print(testset_cells)
print((testset_cells / ((rows+1)*(cols+1)))*100)

In [ ]:
print("y band")

BOSS_cells = 0

for i in range(0, cols):
    for j in range(0, rows): 
        if BOSS[i][j]>=1 and testset[i][j] >=1:
            BOSS_cells += 1

print((BOSS_cells / (testset_cells))*100)

DEEP2_cells = 0

for i in range(0, cols):
    for j in range(0, rows): 
        if DEEP2[i][j]>=1 and testset[i][j] >=1:
            DEEP2_cells += 1

print((DEEP2_cells / (testset_cells))*100)

GAMA_cells = 0

for i in range(0, cols):
    for j in range(0, rows): 
        if GAMA[i][j]>=1 and testset[i][j] >=1:
            GAMA_cells += 1

print((GAMA_cells / (testset_cells))*100)

HSC_cells = 0

for i in range(0, cols):
    for j in range(0, rows): 
        if HSC[i][j]>=1 and testset[i][j] >=1:
            HSC_cells += 1

print((HSC_cells / (testset_cells))*100)

VVDSf02_cells = 0

for i in range(0, cols):
    for j in range(0, rows): 
        if VVDSf02[i][j]>=1 and testset[i][j] >=1:
            VVDSf02_cells += 1

print((VVDSf02_cells / (testset_cells))*100)

zCOSMOS_cells = 0

for i in range(0, cols):
    for j in range(0, rows): 
        if zCOSMOS[i][j]>=5 and testset[i][j] >=1:
            zCOSMOS_cells += 1

print((zCOSMOS_cells / (testset_cells))*100)

In [ ]:
spec_coverage_dict = {'BOSS': [14.489, 11.801, 9.121, 10.069, 8.710, 7.186], 
                      'DEEP2': [68.182, 61.491, 57.330, 67.014, 62.581,63.772], 
                      'GAMA': [26.136, 23.602, 12.052, 30.208, 30.000, 26.627], 
                      'HSC': [75.284, 76.397, 79.153, 83.681, 80.968, 77.276], 
                      'VVDSf02': [79.545, 76.708, 70.033, 66.666, 65.608, 64.071],
                      'zCOSMOS': [40.625, 37.267, 35.179, 34.028, 33.871, 32.036], 
                    }

cov_df = pd.DataFrame.from_dict(spec_coverage_dict, orient='columns').rename(index = {0:'u_band', 1:'g_band', 2:'r_band', 3:'i_band', 4:'z_band', 5:'y_band'})
#cov_df.to_parquet('../paper_data/spec_cov.pq')

In [ ]:
cov_df = pd.read_parquet('../paper_data/spec_cov.pq')
np.sum(cov_df['zCOSMOS'])/6

In [ ]:
cdf = pd.read_parquet('../paper_data/spec_cov.pq')

np.sum(cdf['BOSS'])/6

### Coverage Plot

In [ ]:
cov_df

In [ ]:
fig, axes = plt.subplots(nrows = 3, ncols = 2, figsize =(20, 20) )
fig.tight_layout()
fig.subplots_adjust( wspace = 0)

locs=[1, 2, 3, 4, 5, 6]
line = [0, 1, 2, 3, 4, 5, 6, 7]

for i in range(0, 3):
    axes[i][1].set_xticklabels([ ' ','u', 'g', 'r', 'i', 'z', 'y'])#, labels = [1, 2, 3, 4, 5, 6])
    axes[i][0].set_xticklabels([ ' ','u', 'g', 'r', 'i', 'z', 'y'])#, labels = [1, 2, 3, 4, 5, 6])
    axes[i][1].set_yticklabels([])

for i in range(0, 3):
    for j in range(0, 2):
        axes[i][j].set_ylim(0, 100)
    
        axes[i][j].set_xlim(0.5, 6.5)


axes[0][0].bar(locs, cov_df['BOSS'], alpha=0.5, color ='tab:blue')
axes[0][1].bar(locs, cov_df['GAMA'], alpha=0.5, color ='tab:orange')
axes[0][0].plot(line, (np.sum(cov_df['BOSS'])/6)*np.ones(8), color = 'k', linewidth = 5)
axes[0][1].plot(line, (np.sum(cov_df['GAMA'])/6)*np.ones(8), color = 'k', linewidth = 5, label = 'average')
axes[0][0].set_title('BOSS', y=0.90)
axes[0][1].set_title('GAMA', y=0.90)
axes[0][1].legend(prop={'size':30}, loc=(0.7, 1.1) )

axes[1][0].bar(locs, cov_df['zCOSMOS'], alpha=0.5, color ='tab:green')
axes[1][1].bar(locs, cov_df['DEEP2'], alpha=0.5, color ='tab:red')
axes[1][0].plot(line, (np.sum(cov_df['zCOSMOS'])/6)*np.ones(8), color = 'k', linewidth = 5)
axes[1][1].plot(line, (np.sum(cov_df['DEEP2'])/6)*np.ones(8), color = 'k', linewidth = 5)
axes[1][0].set_title('zCOSMOS', y=0.90)
axes[1][1].set_title('DEEP2', y=0.90)

axes[2][0].bar(locs, cov_df['VVDSf02'], alpha=0.5, color ='tab:purple')
axes[2][1].bar(locs, cov_df['HSC'], alpha=0.5, color ='tab:brown')
axes[2][0].plot(line, (np.sum(cov_df['VVDSf02'])/6)*np.ones(8), color = 'k', linewidth = 5)
axes[2][1].plot(line, (np.sum(cov_df['HSC'])/6)*np.ones(8), color = 'k', linewidth = 5)
axes[2][0].set_title('VVDSf02', y=0.90)
axes[2][1].set_title('HSC', y=0.90)

fig.supxlabel('Bands', y=-0.02, fontsize = 30)
fig.supylabel('Coverage percentage', x=-0.02, fontsize = 30)
fig.suptitle('Percentage of test set covered in each LSST band', y=1.03, size='large')

fig.subplots_adjust(hspace = 0.12, wspace = 0.05)

##

In [ ]:
df = cov_df
df

In [ ]:
new_order = ['BOSS', 'GAMA', 'zCOSMOS', 'DEEP2', 'VVDSf02', 'HSC']
df_reordered = df[new_order]

df_reordered

cov_df = df_reordered
cov_df

In [ ]:
cov_df.loc['u_band']

In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 30

plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)   # fontsize of the figure title
plt.rc('xtick', direction = 'in')
plt.rc('ytick', direction = 'in')
#cmap = sns.color_palette('colorblind')
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.figure(figsize = (10, 10))
# Sample data for 6 clusters
categories = ['BOSS', 'GAMA', 'zCOSMOS', 'DEEP2', 'VVDSf02', 'HSC']
values1 = cov_df.loc['u_band']  # Cluster 1
values2 = cov_df.loc['g_band']   # Cluster 2
values3 = cov_df.loc['r_band']  # Cluster 3
values4 = cov_df.loc['i_band']   # Cluster 4
values5 = cov_df.loc['z_band']  # Cluster 5
values6 = cov_df.loc['y_band']   # Cluster 6

# Bar width
bar_width = 0.12  # Narrower bars to fit more clusters

# Positions of the bars on the y-axis
y_positions = np.arange(len(categories))

# Create the horizontal bar chart
plt.barh(y_positions - bar_width*2.5, values1, bar_width, color='violet', label='u band', edgecolor='black', zorder = 2)
plt.barh(y_positions - bar_width*1.5, values2, bar_width, color='green', label='g band', edgecolor='black')
plt.barh(y_positions - bar_width*0.5, values3, bar_width, color='red', label='r band', edgecolor='black')
plt.barh(y_positions + bar_width*0.5, values4, bar_width, color='darkred', label='i band', edgecolor='black')
plt.barh(y_positions + bar_width*1.5, values5, bar_width, color='orange', label='z band', edgecolor='black')
plt.barh(y_positions + bar_width*2.5, values6, bar_width, color='yellow', label='y band', edgecolor='black')

plt.plot(np.sum(cov_df['BOSS'])/6 * np.ones(3), [-1, 0, 0.35], color = 'k', linestyle = 'dashed')
plt.plot(np.sum(cov_df['GAMA'])/6 * np.ones(3), [-1, 0, 1.35], color = 'k', linestyle = 'dashed')
plt.plot(np.sum(cov_df['zCOSMOS'])/6 * np.ones(3), [-1, 0, 2.35], color = 'k', linestyle = 'dashed')
plt.plot(np.sum(cov_df['DEEP2'])/6 * np.ones(3), [-1, 0, 3.35], color = 'k', linestyle = 'dashed')
plt.plot(np.sum(cov_df['VVDSf02'])/6 * np.ones(3), [-1, 0, 4.35], color = 'k', linestyle = 'dashed')
plt.plot(np.sum(cov_df['HSC'])/6 * np.ones(3), [-1, 0, 5.35], color = 'k', linestyle = 'dashed', label = 'averages', zorder = 1)


plt.ylim(-0.5, 5.5)
plt.xlim(0, 105)
# Add labels and title
plt.xlabel('Percentage Coverage')
#plt.ylabel('Survey')
#plt.title('Coverage')
plt.yticks(y_positions, categories)  # Set category labels for y-axis
plt.legend(prop={'size':15}, loc = 'center right')  # Add legend

# Show the chart
plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sample data for 6 clusters
categories = ['A', 'B', 'C', 'D', 'E', 'F']
values1 = [10, 15, 7, 12, 18, 5]  # Cluster 1
values2 = [8, 13, 5, 10, 15, 7]   # Cluster 2
values3 = [12, 9, 14, 6, 10, 11]  # Cluster 3
values4 = [9, 11, 8, 13, 17, 4]   # Cluster 4
values5 = [14, 10, 12, 9, 16, 6]  # Cluster 5
values6 = [7, 8, 9, 15, 12, 10]   # Cluster 6

# Assign colors to categories
category_colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']

# Set figure size
plt.figure(figsize=(12, 8))

# Bar width
bar_width = 0.12  # Narrower bars to fit more clusters

# Positions of the bars on the y-axis
y_positions = np.arange(len(categories))

# Create the horizontal bar chart
plt.barh(y_positions - bar_width*2.5, values1, bar_width, color='skyblue', label='Cluster 1', edgecolor='black')
plt.barh(y_positions - bar_width*1.5, values2, bar_width, color='orange', label='Cluster 2', edgecolor='black')
plt.barh(y_positions - bar_width*0.5, values3, bar_width, color='green', label='Cluster 3', edgecolor='black')
plt.barh(y_positions + bar_width*0.5, values4, bar_width, color='red', label='Cluster 4', edgecolor='black')
plt.barh(y_positions + bar_width*1.5, values5, bar_width, color='purple', label='Cluster 5', edgecolor='black')
plt.barh(y_positions + bar_width*2.5, values6, bar_width, color='brown', label='Cluster 6', edgecolor='black')

# Add labels and title
plt.xlabel('Values')
plt.ylabel('Categories')
plt.title('Sample Horizontal Bar Chart with 6 Clusters (6 Categories)')

# Customize y-axis tick labels with color
ax = plt.gca()  # Get current axes
ax.set_yticks(y_positions)
# ax.set_yticklabels([f"$\\textcolor{{{color}}}{{{cat}}}$" for cat, color in zip(categories, category_colors)], fontsize=12)

# Enable LaTeX rendering for colored text
plt.rcParams['text.usetex'] = True

plt.legend()  # Add legend
# plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()


In [ ]:
pd.read_parquet('../paper_data/CDE_loss.pq')

In [ ]:
fig, axes = plt.subplots(nrows = 6, ncols = 6, figsize = (80, 40))

for i in range(0, 6):
    for j in range(0, 6):
        file = LSSTerr_only
        axes[i][j].scatter(file['redshift'], file[bands[i]], s=0.5)

for k in range(0, 6):
    deg = invz[k]
    file = getTrainFile(deg)
    for l in range(0, 6):
        axes[l][k].scatter(file['redshift'], file[bands[l]], s=0.5)


mpl.rcParams.update({'font.size':22})

for i in range(0, 6):
    axes[0][i].set_title(invz_names[i])

for i in range(0, 6):
    axes[i][0].set_ylabel(band_names[i])


mpl.rcParams.update({'font.size':30})

fig.supxlabel('Redshift', size = 'x-large')
fig.supylabel('Magnitude', size = 'x-large')
fig.suptitle('Inverse Redshift Incompleteness Degradation', size = 'xx-large')

In [ ]:
specs = ['BOSS', 'GAMA',  'zCOSMOS', 'DEEP2', 'VVDSf02', 'HSC']

In [ ]:
fig, axes = plt.subplots(nrows = 7, ncols = 6, figsize = (30, 24))

for i in range(0, 6):
    for j in range(0, 6):
        file = LSSTerr_only
        axes[i][j].scatter(file['redshift'], file[bands[i]], s=0.5, color = 'pink', alpha = 0.75)
        if i != 6: 
            axes[i][j].set_xticklabels([])
        if j != 0:
            axes[i][j].set_yticklabels([])

for k in range(0, 6):
    deg = specs[k]
    file = removeNans(getTrainFile(deg), bands)
    for l in range(0, 6):
        h = axes[l][k].hist2d(file['redshift'], file[bands[l]], bins = 100, norm = mpl.colors.LogNorm(), label = 'degraded training sets' ) 
        axes[l][k].set_xlim(0, 3)
        axes[l][k].set_ylim(15, 30)
        # axes[l][k].set_xticklabels([])
        # axes[l][k].set_yticklabels([])


for j in range(0,6):
    file = LSSTerr_only
    axes[6][j].hist(file['redshift'], color = 'pink', alpha = 0.95, bins = 30, histtype='step', linewidth = 3)
#     print(len(file['redshift']))
    deg = specs[j]
    file = removeNans(getTrainFile(deg), bands)
    axes[6][j].hist(file['redshift'], color = 'black', alpha = 0.75, bins = 30, histtype='step', linewidth = 3)
#     print(len(file['redshift']))
    if j!=0:
        axes[6][j].set_yticklabels([])
#mpl.rcParams.update({'font.size':22})

axes[0][0].scatter([], [], color = 'pink', label = 'test set')

for i in range(0, 6):
    axes[0][i].set_title(specs[i], fontsize = 40)

for i in range(0, 6):
    axes[i][0].set_ylabel(f'${band_names[i]}$-mag', fontsize=40)
    axes[6][i].set_xlabel(r'$z$', fontsize = 40)
    axes[6][i].set_xticks([0, 1, 2, 3])

cbar_ax = fig.add_axes([ 1.0, 0.1, 0.03, 0.8])
cbar = fig.colorbar(h[3], cax=cbar_ax)
cbar.set_label(label = 'Degraded data density', size = 'large' )
cbar.ax.tick_params(labelsize = 30)
# fig.legend(prop={'size':30})

mpl.rcParams.update({'font.size':30})

#fig.supxlabel('redshift', size = 'xx-large')
#fig.supylabel('15<magnitude<30', size = 'xx-large')
# fig.suptitle('Spectroscopic Survey Degradation', size = 'xx-large')

fig.tight_layout()
fig.subplots_adjust(hspace = 0.12, wspace = 0.05)
plt.savefig('../plots/Spec_Survey_Degradation.png', dpi=300)



In [ ]:
# fig, axes = plt.subplots(nrows = 1, ncols = 7, figsize = (70, 15))

# for i in range(0, 7):
#     file = LSSTerr_only
#     axes[i].scatter(file['redshift'], file[bands[3]], s=0.5, color = 'pink', alpha = 0.75)
#     if i != 0:
#         axes[i].set_xticklabels([])
#         axes[i].set_yticklabels([])
#         axes[i].tick_params(size = 10)
#     if i ==0: 
#         axes[0].tick_params(size = 10, labelsize = 30)
#     if i ==5: 
#         axes[6].scatter([], [], color = 'pink', label = 'test set')
#         axes[6].legend( fontsize = 30)

# for k in range(0, 7):
#     if k != 6:
#         deg = specs[k]
#     else: 
#         deg = 'LSSTerr_only'
#     file = removeNans(getTrainFile(deg), bands)
#     h = axes[k].hist2d(file['redshift'], file[bands[1]], bins = 100, norm = mpl.colors.LogNorm(), label = 'degraded training sets' ) 
#     axes[k].set_xlim(0, 3)
#     axes[k].set_ylim(15, 30)

#         # axes[l][k].set_xticklabels([])
#         # axes[l][k].set_yticklabels([])


# #mpl.rcParams.update({'font.size':22})

# #axes[0].scatter([], [], color = 'pink', label = 'test set')

# # for i in range(0, 6):
# #     axes[0][i].set_title(specs[i], fontsize = 40)

# # for i in range(0, 6):
# #     axes[i][0].set_ylabel(band_names[i], fontsize=40)

# cbar_ax = fig.add_axes([ 1.0, 0.1, 0.01, 0.8])
# cbar = fig.colorbar(h[3], cax=cbar_ax)
# cbar.set_label(label = 'degraded data density',  fontsize = 40)
# cbar.ax.tick_params(labelsize = 30)
# #fig.legend(prop={'size':30})

# #mpl.rcParams.update({'font.size':30})

# axes[0].set_title('pseudo-BOSS (195 galaxies)' , fontsize = 40)
# axes[4].set_title('pseudo-DEEP2 (38,385 objects)', fontsize = 40)
# axes[1].set_title('GAMA (2,099 objects)',fontsize = 40)
# axes[2].set_title('HSC (14,253 objects)',fontsize = 40)
# axes[5].set_title('VVDSf02 (80,845 objects)',fontsize = 40)
# axes[3].set_title('zCOSMOS (32,089 objects)',fontsize = 40)
# axes[6].set_title('LSST Error (30,953 objects)',fontsize = 40)

# fig.supxlabel('redshift, 0<z<3', fontsize = 40)
# fig.supylabel('magnitude, 15<mag<30', fontsize = 40, x=-0.01)
# fig.suptitle('i-band spectroscopic survey degradation', fontsize = 50)

# fig.tight_layout()
# fig.subplots_adjust(hspace = 0, wspace = 0)
# #plt.savefig('Spec_Survey_Degradation.pdf', bbox_inches = 'tight', pad_inches = 0)

In [ ]:
fig, axes = plt.subplots(nrows = 7, ncols = 6, figsize = (30, 24))

for i in range(0, 6):
    for j in range(0, 6):
        file = LSSTerr_only
        axes[i][j].scatter(file['redshift'], file[bands[i]],  color = 'pink', alpha = 0.75)
        if i != 6: 
            axes[i][j].set_xticklabels([])
        if j != 0:
            axes[i][j].set_yticklabels([])

for k in range(0, 6):
    deg = invz[k]
    file = removeNans(getTrainFile(deg), bands)
    for l in range(0, 6):
        h = axes[l][k].hist2d(file['redshift'], file[bands[l]], bins = 100, norm = mpl.colors.LogNorm(), label = 'degraded training sets' ) 
        axes[l][k].set_xlim(0, 3)
        axes[l][k].set_ylim(15, 30)
        #axes[l][k].set_xticklabels([])
        #axes[l][k].set_yticklabels([])

for j in range(0,6):
    file = LSSTerr_only
    axes[6][j].hist(file['redshift'], color = 'pink', alpha = 0.95, bins = 30, histtype='step', linewidth = 3)
#     print(len(file['redshift']))
    deg = invz[j]
    file = removeNans(getTrainFile(deg), bands)
    axes[6][j].hist(file['redshift'], color = 'black', alpha = 0.75, bins = 30, histtype='step', linewidth = 3)
#     print(len(file['redshift']))
    if j!=0:
        axes[6][j].set_yticklabels([])
    
mpl.rcParams.update({'font.size':30})

# axes[0][0].scatter([], [], color = 'pink', label = 'test set')

for i in range(0, 6):
    axes[0][i].set_title(invz_names[i], fontsize = 40)
    axes[6][i].set_xlabel(r'$z$', fontsize = 40)
    axes[6][i].set_xticks([0, 1, 2, 3])

for i in range(0, 6):
    axes[i][0].set_ylabel(f'${band_names[i]}$-mag', fontsize = 40)

# fig.colorbar(h[3], ax=axes, label = 'degraded test sets')
# fig.legend()

cbar_ax = fig.add_axes([ 1.0, 0.1, 0.03, 0.75])
cbar = fig.colorbar(h[3], cax=cbar_ax)
cbar.set_label(label = 'Degraded data density', size = 'large' )
cbar.ax.tick_params(labelsize = 30)
# fig.legend(prop={'size':30})


#fig.supxlabel('0<z<3', size = 'xx-large')
#fig.supylabel('15<magnitude<30', size = 'xx-large')
#fig.suptitle('Inverse Redshift Incompleteness Degradation', size = 'xx-large')

fig.tight_layout()
fig.subplots_adjust(hspace = 0.05, wspace = 0.05)
fig.savefig('../plots/Invz_Degradation.png', dpi=300)



In [ ]:
# fig, axes = plt.subplots(nrows = 1, ncols = 6, figsize = (60, 12))

# for i in range(0, 6):
#     file = LSSTerr_only
#     axes[i].scatter(file['redshift'], file[bands[3]], s=0.5, color = 'pink', alpha = 0.75)
#     if i != 0:
#         axes[i].set_xticklabels([])
#         axes[i].set_yticklabels([])
#         axes[i].tick_params(size = 10)
#     if i ==0: 
#         axes[0].tick_params(size = 10, labelsize = 30)
#     if i ==5: 
#         axes[5].scatter([], [], color = 'pink', label = 'test set')
#         axes[5].legend( fontsize = 30)

# for k in range(0, 6):
#     if k != 5:
#         deg = invz[k]
#     else: 
#         deg = 'LSSTerr_only'
#     file = removeNans(getTrainFile(deg), bands)
#     h = axes[k].hist2d(file['redshift'], file[bands[1]], bins = 100, norm = mpl.colors.LogNorm(), label = 'degraded training sets' ) 
#     axes[k].set_xlim(0, 3)
#     axes[k].set_ylim(15, 30)

#         # axes[l][k].set_xticklabels([])
#         # axes[l][k].set_yticklabels([])


# #mpl.rcParams.update({'font.size':22})

# #axes[0].scatter([], [], color = 'pink', label = 'test set')

# # for i in range(0, 6):
# #     axes[0][i].set_title(specs[i], fontsize = 40)

# # for i in range(0, 6):
# #     axes[i][0].set_ylabel(band_names[i], fontsize=40)

# cbar_ax = fig.add_axes([ 1.0, 0.1, 0.01, 0.8])
# cbar = fig.colorbar(h[3], cax=cbar_ax)
# cbar.set_label(label = 'degraded data density',  fontsize = 40)
# cbar.ax.tick_params(labelsize = 30)
# #fig.legend(prop={'size':30})

# #mpl.rcParams.update({'font.size':30})

# axes[0].set_title('$z_0=0.1$' , size = 40)
# axes[1].set_title('$z_0=0.3$', fontsize = 40)
# axes[2].set_title('$z_0=0.6$',fontsize = 40)
# axes[3].set_title('$z_0=1.0$',fontsize = 40)
# axes[4].set_title('$z_0=1.4$',fontsize = 40)
# axes[5].set_title('LSST Error',fontsize = 40)

# fig.supxlabel('redshift (z)', fontsize = 40)
# fig.supylabel('magnitude', fontsize = 40, x=-0.01)
# fig.suptitle('i-band inverse redshift incompleteness degradation', fontsize = 50)

# fig.tight_layout()
# fig.subplots_adjust(hspace = 0, wspace = 0)
# #plt.savefig('Spec_Survey_Degradation.pdf', bbox_inches = 'tight', pad_inches = 0)

# Visualizing Estimator Outputs 

In [ ]:
true_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
redshift = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")['redshift']
gals = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")['redshift'].index.tolist()

truth_ens = qp.read(true_file)

grid = np.linspace(0, 3, 301)

high_pq = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")['redshift']>1.5
highzs = high_pq.index[high_pq==True].tolist()

print(highzs)
print(gals)

highs = []
c = 0
for i in gals: 
    if i in highzs: 
        highs.append(c)
    c += 1

print(highs)

len(highs) == len(highzs)

In [ ]:
# T_file = "../paper_data/TrainZ/"+deg+'/output_estimate_TrainZ.hdf5'
# C_file = "../paper_data/CMNN/"+deg+'/output_estimate_CMNN.hdf5'
# G_file = "../paper_data/GPz/"+deg+'/output_estimate_GPz.hdf5'
# F_file = "../paper_data/FZBoost/"+deg+'/output_estimate_FZBoost.hdf5'
# P_file = "../paper_data/PZFlow/"+deg+'/output_estimate_pzflow.hdf5'
# GL_file = "../paper_data/GPz_GL/"+deg+'/output_estimate_GPz.hdf5'

# f = h5py.File(C_file)
# f.keys()

# ct = 0
# #truth = truth_ens[ct].pdf(grid)[0]
# truez = redshift[gals[ct]]



In [ ]:
def plotEstOutputs(deg, dim): 
    T_file = "../paper_data/TrainZ/"+deg+'/output_estimate_TrainZ.hdf5'
    C_file = "../paper_data/CMNN/"+deg+'/output_estimate_CMNN.hdf5'
    G_file = "../paper_data/GPz/"+deg+'/output_estimate_GPz.hdf5'
    F_file = "../paper_data/FZBoost/"+deg+'/output_estimate_FZBoost.hdf5'
    P_file = "../paper_data/PZFlow/"+deg+'/output_estimate_pzflow.hdf5'
    GL_file = "../paper_data/GPz_GL/"+deg+'/output_estimate_GPz.hdf5'

    T_ens = qp.read(T_file)
    C_ens = qp.read(C_file)
    G_ens = qp.read(G_file)
    F_ens = qp.read(F_file)
    P_ens = qp.read(P_file)
    GL_ens = qp.read(GL_file)

    fig, axes = plt.subplots(nrows = dim, ncols = dim, figsize = (40, 40))
    ct = 0
    for i in range(0, dim): 
        for j in range(0, dim):
            truth = truth_ens[ct].pdf(grid)[0]
            truez = redshift[gals[ct]]

            TrainZ = T_ens[ct].pdf(grid)[0]
            CMNN = C_ens[ct].pdf(grid)[0]
            #GPz = G_ens[ct].pdf(grid)[0]
            GPzGL = GL_ens[ct].pdf(grid)[0]
            FZBoost = F_ens[ct].pdf(grid)[0]
            PZFlow = P_ens[ct].pdf(grid)[0]

            tr, = axes[i][j].plot(grid, truth, color = 'k', label = 'True Photo-z Posterior')
            z, = axes[i][j].plot(truez*np.ones(31), np.linspace(0, 30, 31), color = 'k', linestyle = 'dashed', label = 'True Spec-z')

            T, = axes[i][j].plot(grid, TrainZ, label = 'TrainZ estimate')
            C, = axes[i][j].plot(grid, CMNN, label = 'CMNN estimate')
            #G, = axes[i][j].plot(grid, GPz, label = 'GPz estimate')
            GL, = axes[i][j].plot(grid, GPzGL, label = 'GPz GL estimate')
            F, = axes[i][j].plot(grid, FZBoost, label = 'FZBoost estimate')
            P, = axes[i][j].plot(grid, PZFlow, label = 'PZFlow estimate' )
            axes[i][j].set_xlim(0, 3)

            ct += 1

    fig.legend(handles = [tr, z, T, C,  GL, F, P])
    fig.suptitle("Estimator outputs for "+deg+' training data', size = 'x-large')
    fig.supxlabel('Redshift')
    fig.supylabel('Probability Density')

In [ ]:
def plotHighZEstOutputs(deg, dim): 
    T_file = "../paper_data/TrainZ/"+deg+'/output_estimate_TrainZ.hdf5'
    C_file = "../paper_data/CMNN/"+deg+'/output_estimate_CMNN.hdf5'
    G_file = "../paper_data/GPz/"+deg+'/output_estimate_GPz.hdf5'
    F_file = "../paper_data/FZBoost/"+deg+'/output_estimate_FZBoost.hdf5'
    P_file = "../paper_data/PZFlow/"+deg+'/output_estimate_pzflow.hdf5'
    GL_file = "../paper_data/GPz_GL/"+deg+'/output_estimate_GPz.hdf5'

    T_ens = qp.read(T_file)
    C_ens = qp.read(C_file)
    G_ens = qp.read(G_file)
    F_ens = qp.read(F_file)
    P_ens = qp.read(P_file)
    GL_ens = qp.read(GL_file)

    fig, axes = plt.subplots(nrows = dim, ncols = dim, figsize = (40, 40))
    ct = 0
    for i in range(0, dim): 
        for j in range(0, dim):
            truth = truth_ens[highs[ct]].pdf(grid)[0]
            truez = redshift[gals[highs[ct]]]

            TrainZ = T_ens[highs[ct]].pdf(grid)[0]
            CMNN = C_ens[highs[ct]].pdf(grid)[0]
            GPzGL = GL_ens[highs[ct]].pdf(grid)[0]
            FZBoost = F_ens[highs[ct]].pdf(grid)[0]
            PZFlow = P_ens[highs[ct]].pdf(grid)[0]

            tr, = axes[i][j].plot(grid, truth, color = 'k', label = 'True Photo-z Posterior')
            z, = axes[i][j].plot(truez*np.ones(31), np.linspace(0, 30, 31), color = 'k', linestyle = 'dashed', label = 'True Spec-z')

            #T, = axes[i][j].plot(grid, TrainZ, label = 'TrainZ estimate')
            C, = axes[i][j].plot(grid, CMNN, label = 'CMNN estimate')
            GL, = axes[i][j].plot(grid, GPzGL, label = 'GPz GL estimate')
            #F, = axes[i][j].plot(grid, FZBoost, label = 'FZBoost estimate')
            P, = axes[i][j].plot(grid, PZFlow, label = 'PZFlow estimate' )
            axes[i][j].set_xlim(0, 3)

            ct += 1

    fig.legend(handles = [tr, z, GL, P, C])#T, C,  GL, F, P])
    fig.suptitle("Estimator outputs for "+deg+' training data', size = 'x-large')
    fig.supxlabel('Redshift')
    fig.supylabel('Probability Density')

In [ ]:
plotHighZEstOutputs('zCOSMOS', 12)

In [ ]:
def plotSpecOutputs(est, dim): 
    fig, axes = plt.subplots(nrows = dim, ncols = dim, figsize = (40, 40))
    ct = 0
    for i in range(0, dim): 
        for j in range(0, dim): 
            truth = truth_ens[ct].pdf(grid)[0]
            truez = redshift[gals[ct]]
            tr, =axes[i][j].plot(grid, truth, color = 'k', label = 'True Photo-z Posterior')
            z, =axes[i][j].plot(truez*np.ones(31), np.linspace(0, 30, 31), color = 'k', linestyle = 'dashed', label = 'True Spec-z')

            B, = axes[i][j].plot(grid, getEns(est, 'BOSS')[ct].pdf(grid)[0], label = 'BOSS')
            D, =axes[i][j].plot(grid, getEns(est, 'DEEP2')[ct].pdf(grid)[0], label = 'DEEP2')
            G, =axes[i][j].plot(grid, getEns(est, 'GAMA')[ct].pdf(grid)[0], label = 'GAMA')
            H, =axes[i][j].plot(grid, getEns(est, 'HSC')[ct].pdf(grid)[0], label = 'HSC')
            V, =axes[i][j].plot(grid, getEns(est, 'VVDSf02')[ct].pdf(grid)[0], label = 'VVDSf02')
            Z, =axes[i][j].plot(grid, getEns(est, 'zCOSMOS')[ct].pdf(grid)[0], label = 'zCOSMOS')
            ct += 1
    
    fig.legend(handles = [tr, z, B, D, G, H, V, Z])
    fig.suptitle('Comparison of '+est+' Outputs for Spectroscopic Survey Training')
    fig.supxlabel('Redshift')
    fig.supylabel('Probability Density')

In [ ]:
def plotInvzOutputs(est, dim): 
    fig, axes = plt.subplots(nrows = dim, ncols = dim, figsize = (40, 40))
    ct = 0
    for i in range(0, dim): 
        for j in range(0, dim): 
            truth = truth_ens[ct].pdf(grid)[0]
            truez = redshift[gals[ct]]
            tr, =axes[i][j].plot(grid, truth, color = 'k', label = 'True Photo-z Posterior')
            z, =axes[i][j].plot(truez*np.ones(31), np.linspace(0, 30, 31), color = 'k', linestyle = 'dashed', label = 'True Spec-z')

            z01, = axes[i][j].plot(grid, getEns(est, 'invz=0.1')[ct].pdf(grid)[0], label = '$z_0=0.1$')
            z03, = axes[i][j].plot(grid, getEns(est, 'invz=0.3')[ct].pdf(grid)[0], label = '$z_0=0.3$')
            z06, =axes[i][j].plot(grid, getEns(est, 'invz=0.6')[ct].pdf(grid)[0], label = '$z_0=0.6$')
            z10, =axes[i][j].plot(grid, getEns(est, 'invz=1.0')[ct].pdf(grid)[0], label = '$z_0=1.0$')
            z14, =axes[i][j].plot(grid, getEns(est, 'invz=1.4')[ct].pdf(grid)[0], label = '$z_0=1.4$')
            n, =axes[i][j].plot(grid, getEns(est, 'LSSTerr_only')[ct].pdf(grid)[0], label = 'LSST error only')
            ct += 1
    
    fig.legend(handles = [tr, z, z01, z03, z06, z10, z14, n])
    fig.suptitle('Comparison of '+est+' Outputs for Inverse Redshift Incompleteness Training')
    fig.supxlabel('Redshift')
    fig.supylabel('Probability Density')

In [ ]:
plotInvzOutputs('GPz_GL', 5)

In [ ]:
plotInvzOutputs('CMNN', 5)

In [ ]:
plotSpecOutputs('FZBoost', 5)

# Distribution-to-distribution Statistics

## Histograms

In [ ]:
def getStatsFile(est, stat):
    try:
        res = pd.read_parquet(est+'/'+stat+'_stats_tq.pq')
        print(f"used new stats for {est}, {stat}")
    except:
        res = pd.read_parquet(est+'/'+stat+'_stats.pq')
    return res

In [ ]:
def makeStatHists(stat, deg, min, max, binnum, axes):

    import matplotlib.pyplot as plt
    import pandas as pd 
    
    grid = np.linspace(min, max, binnum)

    axes.hist(getStatsFile('TrainZ', stat)[deg], label = 'TrainZ', bins = grid, alpha = 0.5)
    axes.hist(getStatsFile('CMNN', stat)[deg], label = 'CMNN', bins = grid, alpha = 0.5)
    #axes.hist(getStatsFile('GPz', stat)[deg], label = 'GPz', bins = grid, alpha = 0.5)
    axes.hist(getStatsFile('GPz_GL', stat)[deg], label = 'GPz_GL', bins = grid, alpha = 0.5)
    axes.hist(getStatsFile('FZBoost', stat)[deg], label = 'FZBoost', bins = grid, alpha = 0.5)
    axes.hist(getStatsFile('PZFlow', stat)[deg], label = 'PZFlow', bins = grid, alpha = 0.5)
        
    axes.legend()
    #axes.set_xlim(min, max)
    #axes.set_xscale('log')
    
    #axes.set_yscale('log')
    #axes.ylim(0, 10000)
    
    axes.set_title(stat)

In [ ]:
# choose training set degradation by setting trainset = [desired training data]
# note: I will make the titles look nicer once we're ready to choose which plots to go in the paper! 

trainset = 'DEEP2'

In [ ]:
fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (24, 16) )


makeStatHists('ks', trainset, 0, 1, 100, axes[0][0])
makeStatHists('cvm', trainset, 0, 35, 100, axes[0][1])
makeStatHists('rmse', trainset, 0, 20, 100, axes[1][0])
makeStatHists('kld', trainset, 0, 0.05, 100, axes[1][1])


fig.supxlabel('Statistic Value')
fig.supylabel('Bin Count')
fig.suptitle("Distribution of statistic values for "+trainset+" training set", size = 'xx-large')

In [ ]:
def makeLogStatHists(stat, deg, binnum, axes):

    axes.hist(np.log(getStatsFile('TrainZ', stat)[deg]), label = 'TrainZ', bins = binnum, alpha = 0.3)
    axes.hist(np.log(getStatsFile('CMNN', stat)[deg]), label = 'CMNN', bins = binnum, alpha = 0.3)
    #axes.hist(np.log(getStatsFile('GPz', stat)[deg]), label = 'GPz', bins = binnum, alpha = 0.3)
    axes.hist(np.log(getStatsFile('GPz_GL', stat)[deg]), label = 'GPz_GL', bins = binnum, alpha = 0.3)
    axes.hist(np.log(getStatsFile('FZBoost', stat)[deg]), label = 'FZBoost', bins = binnum, alpha = 0.3)
    axes.hist(np.log( removeNans(getStatsFile('PZFlow', stat), [deg] ))[deg], label = 'PZFlow', bins = binnum, alpha = 0.3)
        
    axes.legend()
    #axes.set_xlim(min, max)
    #axes.set_xscale('log')
    axes.set_yscale('log')
    #axes.ylim(0, 10000)
    
    axes.set_title(stat)

In [ ]:
fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (24, 16) )


makeLogStatHists('ks', trainset, 50, axes[0][0])
makeLogStatHists('cvm', trainset, 50, axes[0][1])
makeLogStatHists('rmse', trainset, 50, axes[1][0])
makeLogStatHists('kld', trainset, 50, axes[1][1])


fig.supxlabel('Log Statistic Value')
fig.supylabel('Log Bin Count')
fig.suptitle("Distribution of statistic values for "+trainset+" training set", size = 'xx-large')

## Quantile Plots

In [ ]:
def helpLogPlots(deg, stat, grid):
    t = plt.hist(getStatsFile('TrainZ', stat)[deg], bins = grid, color = 'white')
    c = plt.hist(getStatsFile('CMNN', stat)[deg], bins = grid, color = 'white')
    #g = plt.hist(getStatsFile('GPz', stat)[deg], bins = grid, color = 'white')
    gl = plt.hist(getStatsFile('GPz_GL', stat)[deg], bins = grid, color = 'white')
    f = plt.hist(getStatsFile('FZBoost', stat)[deg], bins = grid, color = 'white')
    p = plt.hist(removeNans(getStatsFile('PZFlow', stat),[deg])[deg] , bins = grid, color = 'white')
    return (t[0], t[1]), (c[0], c[1]), (gl[0], gl[1]), (f[0], f[1]), (p[0], p[1])

In [ ]:
def makeLogStatPlots(stat, deg, grid, axes):

    # t = plt.hist(getStatsFile('TrainZ', stat)[deg], bins = grid, color = 'white')
    # c = plt.hist(getStatsFile('CMNN', stat)[deg], bins = grid, color = 'white')
    # g = plt.hist(getStatsFile('GPz', stat)[deg], bins = grid, color = 'white')
    # f = plt.hist(getStatsFile('FZBoost', stat)[deg], bins = grid, color = 'white')
    # p = plt.hist(removeNans(getStatsFile('PZFlow', stat),[deg])[deg] , bins = grid, color = 'white')
    t, c, gl, f, p = helpLogPlots(deg, stat, grid)
    #plt.show()

    tx = t[1].tolist()
    tx.pop(-1)
    cx = c[1].tolist()
    cx.pop(-1)
    # gx = g[1].tolist()
    # gx.pop(-1)
    glx = gl[1].tolist()
    glx.pop(-1)
    fx = f[1].tolist()
    fx.pop(-1)
    px = p[1].tolist()
    px.pop(-1)

    td = []
    cd = []
    # gd = []
    gld = []
    fd = []
    pd = []

    val = 0
    for i in t[0]: 
        val += i
        td.append(val)
    td = td/np.sum(t[0])

    val = 0
    for i in c[0]: 
        val += i
        cd.append(val)
    cd = cd/np.sum(c[0])    
    
    val = 0
    for i in gl[0]: 
        val += i
        gld.append(val)
    gld = gld/np.sum(gl[0])

    val = 0
    for i in f[0]: 
        val += i
        fd.append(val)
    fd = fd/np.sum(f[0])

    val = 0
    for i in p[0]: 
        val += i
        pd.append(val)
    pd = pd/np.sum(p[0])

    axes.plot(np.asarray(tx) + (tx[1] - tx[0])/2, td, label = 'TrainZ')
    axes.plot(np.asarray(cx) + (cx[1] - cx[0])/2, cd, label = 'CMNN')
    # axes.plot(np.asarray(gx) + (gx[1] - gx[0])/2, gd, label = 'GPz')
    axes.plot(np.asarray(glx) + (glx[1] - glx[0])/2, gld, label = 'GPz GL')
    axes.plot(np.asarray(fx) + (fx[1] - fx[0])/2, fd, label = 'FZBoost')
    axes.plot(np.asarray(px) + (px[1] - px[0])/2,  pd, label = 'PZFlow')
        
    axes.legend()
    #axes.set_xlim(min, max)
    #axes.set_xscale('log')
    axes.set_yscale('log')
    #axes.set_ylim(0, 1e5)
    
    axes.set_title(stat)

In [ ]:
trainset = 'LSSTerr_only'

In [ ]:
# fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (24, 24) )


# makeLogStatPlots('ks', trainset, np.logspace(-3, 0.05, 100), axes[0][0])
# makeLogStatPlots('cvm', trainset, np.logspace(-3, 1.65, 100), axes[0][1])
# makeLogStatPlots('wdist', trainset, np.logspace(-7, 1.2, 100), axes[1][0])
# makeLogStatPlots('kld', trainset, np.logspace(-5, 0.5, 100), axes[1][1])
# axes[1][1].set_xlim(-0.1, 2.3)
# axes[1][1].set_ylim(1e-1, 10**(0.08))

# fig.supxlabel('Log Statistic Value')
# fig.supylabel('Fraction of Data')
# fig.suptitle("Quantile plots of statistic values for "+trainset+" training set", size = 'xx-large')

In [ ]:
def getSpecQuants(est, stat):
    file = getStatsFile(est, stat)
    data_dict = {'BOSS': [], 
                 'DEEP2': [], 
                 'GAMA': [], 
                 'HSC': [], 
                 'VVDSf02': [], 
                 'zCOSMOS': [],
                 #'LSSTerr_only': []
                 }
    
    for key in data_dict: 
        data = np.log(removeNans(file, [key])[key])
#         data = removeNans(file, [key])[key]
        counts = np.histogram(data, bins=100)[0]
        bins = np.histogram(data, bins=100)[1]
        grid = ((bins[:-1] + bins[1:]) / 2).tolist()

        quants = []
        val = 0
        for i in counts: 
            val += i
            quants.append(val)
        
        data_dict[key] = [grid, quants]
    
    return data_dict
    

In [ ]:
def getInvzQuants(est, stat):
    file = getStatsFile(est, stat)
    data_dict = {'invz=0.1': [], 
                 'invz=0.3': [], 
                 'invz=0.6': [], 
                 'invz=1.0': [], 
                 'invz=1.4': [], 
                 'LSSTerr_only': []
                 }
    
    for key in data_dict: 
        data = np.log(removeNans(file, [key])[key])
#         data = removeNans(file, [key])[key]
        counts = np.histogram(data, bins=100)[0]
        bins = np.histogram(data,bins=100)[1]
        grid = ((bins[:-1] + bins[1:]) / 2).tolist()

        quants = []
        val = 0
        for i in counts: 
            val += i
            quants.append(val)
        
        data_dict[key] = [grid, quants]
    
    return data_dict
    

In [ ]:
def getInvzQuantsWithCI(est, stat, n_bootstrap=200, rng_seed=42):
    """Like getInvzQuants but also returns bootstrap 16th/84th percentile CI bands.
    Returns data_dict[key] = [grid, quants, lo_frac, hi_frac]
    where lo_frac/hi_frac are normalized CDFs (0-1) at the 16th/84th bootstrap percentiles.
    """
    file = getStatsFile(est, stat)
    data_dict = {'invz=0.1': [],
                 'invz=0.3': [],
                 'invz=0.6': [],
                 'invz=1.0': [],
                 'invz=1.4': [],
                 'LSSTerr_only': []}
    rng = np.random.default_rng(rng_seed)
    for key in data_dict:
        data = np.log(removeNans(file, [key])[key].values)
        # Fix bin edges from original data so bootstrap samples align
        _, bin_edges = np.histogram(data, bins=100)
        counts = np.histogram(data, bins=bin_edges)[0]
        # Grid: same offset convention as getInvzQuants
        grid = ((bin_edges[:-1] + bin_edges[1:]) / 2).tolist()
        # Original cumulative counts (matches getInvzQuants output)
        quants = np.cumsum(counts).tolist()
        # Bootstrap: resample with replacement, compute normalized CDF each time
        boot_cdfs = np.zeros((n_bootstrap, 100))
        for b in range(n_bootstrap):
            sample = rng.choice(data, size=len(data), replace=True)
            bc = np.histogram(sample, bins=bin_edges)[0]
            bc_cum = np.cumsum(bc).astype(float)
            tot = bc_cum[-1] if bc_cum[-1] > 0 else 1.0
            boot_cdfs[b] = bc_cum / tot
        lo = np.percentile(boot_cdfs, 16, axis=0).tolist()
        hi = np.percentile(boot_cdfs, 84, axis=0).tolist()
        data_dict[key] = [grid, quants, lo, hi]
    return data_dict

In [ ]:
def makeQuantPlots(stat, kind):
    fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize =(24, 24) )
    if kind == 'spec': 
        name = 'Spectroscopic Survey Degradation'
        degs = ['BOSS', 
                'DEEP2', 
                'GAMA', 
                'HSC', 
                'VVDSf02', 
                'zCOSMOS'
                ]
        c_dict = getSpecQuants('CMNN', stat)
        gl_dict = getSpecQuants('GPz_GL', stat)
        f_dict = getSpecQuants('FZBoost', stat)
        p_dict = getSpecQuants('PZFlow', stat)
    elif kind == 'invz':
        name = 'Inverse Redshift Incompleteness Degradation'
        degs = ['invz=0.1', 
                'invz=0.3',
                'invz=0.6', 
                'invz=1.0', 
                'invz=1.4', 
                'LSSTerr_only'
                ]
        c_dict = getInvzQuants('CMNN', stat)
        gl_dict = getInvzQuants('GPz_GL', stat)
        f_dict = getInvzQuants('FZBoost', stat)
        p_dict = getInvzQuants('PZFlow', stat)
    
    for key in c_dict: 
        axes[0][0].plot(c_dict[key][0], c_dict[key][1])#, label = key)
    for key in gl_dict: 
        axes[0][1].plot(gl_dict[key][0], gl_dict[key][1])#, label = key)
    for key in f_dict: 
        axes[1][0].plot(f_dict[key][0], f_dict[key][1])#, label = key)
    for key in p_dict: 
        axes[1][1].plot(p_dict[key][0], p_dict[key][1], label = key)

    axes[0][0].set_title('CMNN')
    #axes[0][0].set_xlim(-12, 2)
    axes[0][1].set_title('GPz GL')
    #axes[0][1].set_xlim(-12, 2)
    axes[1][0].set_title('FZBoost')
    #axes[1][0].set_xlim(-12, 2)
    axes[1][1].set_title('PZFlow')
    #axes[1][1].set_xlim(-12, 2)

    fig.suptitle('Quantile plots of '+stat+' results for '+name)
    fig.supxlabel('Log '+stat)
    fig.supylabel('Number of galaxies output')
    fig.legend()

In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 30

plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)   # fontsize of the figure title
plt.rc('xtick', direction = 'in')
plt.rc('ytick', direction = 'in')
#cmap = sns.color_palette('colorblind')
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
t_dict = getInvzQuantsWithCI('TrainZ', 'ks')

stats = getStatsFile('TrainZ', 'ks')

In [ ]:
len(stats)

In [ ]:
plt.hist(stats['invz=0.6'], bins = 200)
# plt.yscale('log')
plt.xlabel('KS')
plt.title('TrainZ, KS')
plt.show()

In [ ]:
t_dict = getSpecQuants('TrainZ', 'ks')
t_dict.keys()

In [ ]:
def makeBigQuantPlots(kind):
    fig, axes = plt.subplots(nrows = 5, ncols = 5, figsize =(25, 25) )
    stats = ['cvm', 'ks', 'rmse', 'kld', 'wdist']
    col = 0
    norm_total = 31247

    for stat in stats:

        if kind == 'spec': 
            name = 'Spectroscopic Survey Degradation'
            degs = ['BOSS', 
                    'DEEP2', 
                    'GAMA', 
                    'HSC', 
                    'VVDSf02', 
                    'zCOSMOS'
                    ]
            t_dict = getSpecQuants('TrainZ', stat)
            c_dict = getSpecQuants('CMNN', stat)
            gl_dict = getSpecQuants('GPz_GL', stat)
            f_dict = getSpecQuants('FZBoost', stat)
            p_dict = getSpecQuants('PZFlow', stat)
            colors = ['red', 'darkorange', 'goldenrod', 'green', 'blue', 'black']


            cct = 0
            for key in c_dict: 
                if key == 'LSSTerr_only':
                    axes[1][col].plot(t_dict[key][0], np.asarray(c_dict[key][1])/np.max(c_dict[key][1]),  color=colors[cct])
                else:
                    axes[1][col].plot(c_dict[key][0], np.asarray(c_dict[key][1])/np.max(c_dict[key][1]),  color=colors[cct])
                axes[1][col].tick_params(axis='both')
                cct += 1
            
            cct = 0
            for key in gl_dict: 
                if key == 'LSSTerr_only':
                    axes[2][col].plot(t_dict[key][0], np.asarray(gl_dict[key][1])/np.max(gl_dict[key][1]),  color=colors[cct])
                else:
                    axes[2][col].plot(gl_dict[key][0], np.asarray(gl_dict[key][1])/np.max(gl_dict[key][1]),  color=colors[cct])
                axes[2][col].tick_params(axis='both')
                cct += 1
            cct = 0
            for key in f_dict: 
                if key == 'LSSTerr_only':
                    axes[3][col].plot(t_dict[key][0], np.asarray(f_dict[key][1])/np.max(f_dict[key][1]),  color=colors[cct])
                else:
                    axes[3][col].plot(f_dict[key][0], np.asarray(f_dict[key][1])/np.max(f_dict[key][1]),  color=colors[cct])
                axes[3][col].tick_params(axis='both')
                cct += 1
            cct = 0
            for key in p_dict: 

                if key == 'LSSTerr_only':
                    axes[4][col].plot(p_dict[key][0], np.asarray(p_dict[key][1])/np.max(p_dict[key][1]),  color=colors[cct])
                    axes[4][col].plot(p_dict[key][0], np.asarray(p_dict[key][1])/ norm_total,  '--',  color=colors[cct])
                else: 
                    axes[4][col].plot(p_dict[key][0], np.asarray(p_dict[key][1])/np.max(p_dict[key][1]),  color=colors[cct])
                    axes[4][col].plot(p_dict[key][0], np.asarray(p_dict[key][1]) / norm_total, '--',  color=colors[cct] )

                axes[4][col].tick_params(axis='both')
                cct += 1


        elif kind == 'invz':
            name = 'Inverse Redshift Incompleteness Degradation'
            degs = ['invz=0.1', 
                    'invz=0.3',
                    'invz=0.6', 
                    'invz=1.0', 
                    'invz=1.4', 
                    'LSSTerr_only'
                    ]
            colors = ['red', 'darkorange', 'goldenrod', 'green', 'blue', 'black']

            t_dict = getInvzQuants('TrainZ', stat)
            c_dict = getInvzQuants('CMNN', stat)
            gl_dict = getInvzQuants('GPz_GL', stat)
            f_dict = getInvzQuants('FZBoost', stat)
            p_dict = getInvzQuants('PZFlow', stat)

            cct = 0
            for key in p_dict:
                norm_p = np.max(np.asarray(p_dict[key][1]))
#                 print(norm_p)
                axes[4][col].plot(p_dict[key][0], np.asarray(p_dict[key][1]) / norm_p, color=colors[cct])
#                 axes[4][col].plot(p_dict[key][0], np.asarray(p_dict[key][1]) / norm_total,'--', color=colors[cct])
#                 axes[4][col].fill_between(p_dict[key][0],
#                                            np.asarray(p_dict[key][2]),
#                                            np.asarray(p_dict[key][3]),
#                                            color=colors[cct], alpha=0.15)
                axes[4][col].tick_params(axis='both')
#                 print(np.asarray(p_dict[key][3]) - np.asarray(p_dict[key][2]))
                cct += 1
            cct = 0
            for key in c_dict:
                norm_c = np.max(np.asarray(c_dict[key][1]))
                axes[1][col].plot(c_dict[key][0], np.asarray(c_dict[key][1]) / norm_c, color=colors[cct])
#                 axes[1][col].fill_between(c_dict[key][0],
#                                            np.asarray(c_dict[key][2]),
#                                            np.asarray(c_dict[key][3]),
#                                            color=colors[cct], alpha=0.15)
#                 print(np.asarray(c_dict[key][2]))
#                 print(np.asarray(c_dict[key][3]))
                axes[1][col].tick_params(axis='both')
                cct += 1
            cct = 0
            for key in gl_dict:
                norm_gl = np.max(np.asarray(gl_dict[key][1]))
                axes[2][col].plot(gl_dict[key][0], np.asarray(gl_dict[key][1]) / norm_gl, color=colors[cct])
#                 axes[2][col].fill_between(gl_dict[key][0],
#                                            np.asarray(gl_dict[key][2]),
#                                            np.asarray(gl_dict[key][3]),
#                                            color=colors[cct], alpha=0.15)
                axes[2][col].tick_params(axis='both')
                cct += 1
            cct = 0
            for key in f_dict:
                norm_f = np.max(np.asarray(f_dict[key][1]))
                axes[3][col].plot(f_dict[key][0], np.asarray(f_dict[key][1]) / norm_f, color=colors[cct])
#                 axes[3][col].fill_between(f_dict[key][0],
#                                            np.asarray(f_dict[key][2]),
#                                            np.asarray(f_dict[key][3]),
#                                            color=colors[cct], alpha=0.15)
                axes[3][col].tick_params(axis='both')
                cct += 1


        if kind =='invz':
            labels = ['$z_0=0.1$','$z_0=0.3$', '$z_0=0.6$', '$z_0=1.0$', '$z_0=1.4$', 'LSST Error \n Model only']
            ct=0

        cct = 0
        for key in t_dict: 
            if col == 4:
                if kind =='invz':
                    norm_t = np.max(np.asarray(t_dict[key][1]))
                    axes[0][col].plot(t_dict[key][0], np.asarray(t_dict[key][1]) / norm_t, label=labels[ct], color=colors[cct])
#                     axes[0][col].fill_between(t_dict[key][0],
#                                                np.asarray(t_dict[key][2]),
#                                                np.asarray(t_dict[key][3]),
#                                                color=colors[cct], alpha=0.15)
                    
                else:
                    if key == 'LSSTerr_only': 
                        axes[0][col].plot(t_dict[key][0], np.asarray(t_dict[key][1])/np.max(t_dict[key][1]), color=colors[cct], label = 'LSST Error Only')
                    elif key in ['BOSS', 'DEEP2']:
                        axes[0][col].plot(t_dict[key][0], np.asarray(t_dict[key][1])/np.max(t_dict[key][1]), label = "pseudo-"+key, color=colors[cct])
                    else: 
                        print('train-z, else, not plotting range')
                        axes[0][col].plot(t_dict[key][0], np.asarray(t_dict[key][1])/np.max(t_dict[key][1]), label = key, color=colors[cct])
                cct += 1
                axes[0][col].legend( prop = {'size': 25}, frameon=False, handlelength=0.5)
            else:
                if kind =='invz':
                    norm_t = np.max(np.asarray(t_dict[key][1]))
                    axes[0][col].plot(t_dict[key][0], np.asarray(t_dict[key][1]) / norm_t, color=colors[cct])
#                     axes[0][col].fill_between(t_dict[key][0],
#                                                np.asarray(t_dict[key][2]),
#                                                np.asarray(t_dict[key][3]),
#                                                color=colors[cct], alpha=0.5)
                else:
                    if key == 'LSSTerr_only':
                        axes[0][col].plot(t_dict[key][0], np.asarray(t_dict[key][1])/np.max(t_dict[key][1]), color=colors[cct])
                    else: 
                        axes[0][col].plot(t_dict[key][0], np.asarray(t_dict[key][1])/np.max(t_dict[key][1]), color=colors[cct])
                cct+=1  
            axes[0][col].tick_params(axis='both')
            if kind =='invz':
                ct += 1

        
        if col==0:
            xlow, xhigh = 1,6.5
            axes[0][col].set_xlim(xlow, xhigh)
            axes[1][col].set_xlim(xlow, xhigh)
            axes[2][col].set_xlim(xlow, xhigh)
            axes[3][col].set_xlim(xlow, xhigh)
            axes[4][col].set_xlim(xlow, xhigh)
        elif col==1: 
            xlow, xhigh = -1.4,0.2
            axes[0][col].set_xlim(xlow, xhigh)
            axes[1][col].set_xlim(xlow, xhigh)
            axes[2][col].set_xlim(xlow, xhigh)
            axes[3][col].set_xlim(xlow, xhigh)
            axes[4][col].set_xlim(xlow, xhigh)
        elif col==2: 
            xlow, xhigh = -3.5 ,4.1
            axes[0][col].set_xlim(xlow, xhigh)
            axes[1][col].set_xlim(xlow, xhigh)
            axes[2][col].set_xlim(xlow, xhigh)
            axes[3][col].set_xlim(xlow, xhigh)
            axes[4][col].set_xlim(xlow, xhigh)
        elif col==3: 
            xlow, xhigh = -10.5 ,1.5
            axes[0][col].set_xlim(xlow, xhigh)
            axes[1][col].set_xlim(xlow, xhigh)
            axes[2][col].set_xlim(xlow, xhigh)
            axes[3][col].set_xlim(xlow, xhigh)
            axes[4][col].set_xlim(xlow, xhigh)
        elif col==4: 
            xlow, xhigh = -6 ,1.8
            axes[0][col].set_xlim(xlow, xhigh)
            axes[1][col].set_xlim(xlow, xhigh)
            axes[2][col].set_xlim(xlow, xhigh)
            axes[3][col].set_xlim(xlow, xhigh)
            axes[4][col].set_xlim(xlow, xhigh)

        col += 1

    mpl.rcParams.update({'font.size':20})
    
    axes[0][0].set_title('TrainZ', loc='left', x = 0.05, y = 0.85)
    axes[1][0].set_title('CMNN', loc='left', x = 0.05, y = 0.85)
    axes[2][0].set_title('GPz GL', loc='left', x = 0.05, y = 0.85)
    axes[3][0].set_title('FZBoost', loc='left', x = 0.05, y = 0.85)
    axes[4][0].set_title('PZFlow', loc='left', x = 0.05, y = 0.85)

    axes[4][0].set_xlabel('log(CvM)')
    axes[4][1].set_xlabel('log(KS)')
    axes[4][2].set_xlabel('log(RMSE)')
    axes[4][3].set_xlabel('log(KLD)')
    axes[4][4].set_xlabel('log(W. Dist)')

    for i in range(0, 5):
        for j in range(0, 5):
            if i != 4: 
                axes[i][j].set_xticklabels([])
            if j != 0: 
                axes[i][j].set_yticklabels([])
        
    mpl.rcParams.update({'font.size':25})
    #fig.suptitle('Summary Statistic Quantile Plots for '+name)
    #fig.supxlabel('Log statistic value')
    fig.supylabel('Fraction of galaxies $\leq$ stat.', x=-0.01)

    fig.tight_layout()
    fig.subplots_adjust(hspace = 0.05, wspace = 0.05)
    if kind == "invz":
        plt.savefig('../plots/Invz_D2D.pdf', bbox_inches = 'tight', pad_inches = 0)
    else:    
        plt.savefig('../plots/Spec_Dist-to-dist.pdf', bbox_inches = 'tight', pad_inches = 0)
    

In [ ]:
def makeOneBigQuantPlots(kind):
    fig, axes = plt.subplots(nrows = 1, ncols = 5, figsize =(32, 8) )
    stats = ['cvm', 'ks', 'rmse', 'kld']
    col = 0

    stat = 'kld'

    if kind == 'spec': 
        name = 'Spectroscopic Survey Degradation'
        degs = ['BOSS', 
                'DEEP2', 
                'GAMA', 
                'HSC', 
                'VVDSf02', 
                'zCOSMOS',
                'LSSTerr_only'
                ]
        t_dict = getSpecQuants('TrainZ', stat)
        c_dict = getSpecQuants('CMNN', stat)
        gl_dict = getSpecQuants('GPz_GL', stat)
        f_dict = getSpecQuants('FZBoost', stat)
        p_dict = getSpecQuants('PZFlow', stat)


        for key in p_dict: 

            if key == 'LSSTerr_only':
                axes[4].plot(p_dict[key][0], np.asarray(p_dict[key][1])/31247, color = 'k')
            else:
                axes[4].plot(p_dict[key][0], np.asarray(p_dict[key][1])/31247, label = key)
            axes[4].tick_params(axis='both', labelsize=15)
            axes[4].legend(prop = {'size': 17})
            axes[4].set_yticklabels([])
            axes[4].set_xticklabels([])
            axes[4].set_xlim(-15, 3)

        for key in c_dict: 
            if key == 'LSSTerr_only':
                axes[1].plot(c_dict[key][0], np.asarray(c_dict[key][1])/31247, color = 'k')
            else:
                axes[1].plot(c_dict[key][0], np.asarray(c_dict[key][1])/31247)
            axes[1].tick_params(axis='both', labelsize=15)
            axes[1].set_yticklabels([])
            axes[1].set_xticklabels([])
            axes[1].set_xlim(-15, 3)

        for key in gl_dict: 
            if key == 'LSSTerr_only':
                axes[2].plot(c_dict[key][0], np.asarray(gl_dict[key][1])/31247, color = 'k')
            else:
                axes[2].plot(gl_dict[key][0], np.asarray(gl_dict[key][1])/31247)
            axes[2].tick_params(axis='both', labelsize=15)
            axes[2].set_yticklabels([])
            axes[2].set_xticklabels([])
            axes[2].set_xlim(-15, 3)

        for key in f_dict: 
            if key == 'LSSTerr_only':
                axes[3].plot(f_dict[key][0], np.asarray(f_dict[key][1])/31247, color = 'k')
            else:
                axes[3].plot(f_dict[key][0], np.asarray(f_dict[key][1])/31247)
            axes[3].tick_params(axis='both', labelsize=15)
            axes[3].set_yticklabels([])
            axes[3].set_xticklabels([])
            axes[3].set_xlim(-15, 3)

        for key in t_dict: 
            if key == 'LSSTerr_only':
                axes[0].plot(t_dict[key][0], np.asarray(t_dict[key][1])/31247, color = 'k')
            else:
                axes[0].plot(t_dict[key][0], np.asarray(t_dict[key][1])/31247)
            axes[0].tick_params(axis='both', labelsize=15)
            axes[0].set_xlim(-15, 3)



    mpl.rcParams.update({'font.size':20})
    
    axes[0].set_title('TrainZ')
    axes[1].set_title('CMNN')
    axes[2].set_title('GPz GL')
    axes[3].set_title('FZBoost')
    axes[4].set_title('PZFlow')
        
    mpl.rcParams.update({'font.size':25})
    fig.suptitle('KLD Quantile Plots for '+name)
    fig.supxlabel('Log KLD value')
    fig.supylabel('Fraction of galaxies estimated on ')

    fig.tight_layout()
    fig.subplots_adjust(hspace = 0, wspace = 0)

    if kind == "invz":
        plt.savefig('../plots/Invz_D2D.pdf', dpi=300)

    #plt.savefig('../plots/Spec_D2D.pdf', dpi=300)
    

In [ ]:
# makeOneBigQuantPlots('spec')

In [ ]:
makeBigQuantPlots('invz')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# --- get these however you already do ---
t_dict  = getSpecQuants('TrainZ',   'kld')
c_dict  = getSpecQuants('CMNN',     'kld')
gl_dict = getSpecQuants('GPz_GL',   'kld')
f_dict  = getSpecQuants('FZBoost',  'kld')
p_dict  = getSpecQuants('PZFlow',   'kld')

# -------------------- PLOTTING --------------------
fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(25, 8))

# consistent style
norm = 31247.  # your normalization

algo_dicts = [t_dict, c_dict, gl_dict, f_dict, p_dict]
titles     = ['TrainZ', 'CMNN', 'GPz GL', 'FZBoost', 'PZFlow']

for i, (dd, title) in enumerate(zip(algo_dicts, titles)):
    for key in dd:
        if key == 'LSSTerr_only':
            axes[i].plot(dd[key][0], np.asarray(dd[key][1]) / np.max(dd[key][1]), color='k')
        else:
            axes[i].plot(dd[key][0], np.asarray(dd[key][1]) / np.max(dd[key][1]), label=key)

    axes[i].set_xlim(-10.5, 3)           # log-RMSE range
    axes[i].tick_params(axis='both')

    # remove duplicate ticks except the first panel
    if i != 0:
        axes[i].set_yticklabels([])
#     axes[i].set_xticklabels([])

    axes[i].set_title(title)

# only show legend on last panel
axes[-1].legend(fontsize = 18, ncol = 1)

# mpl.rcParams.update({'font.size':22})
fig.suptitle('log(KLD) Quantile Plots for Spectroscopic Survey Degradation')
fig.supxlabel('log(KLD)', y=0.02)
fig.supylabel('Fraction of galaxies estimated on')

fig.tight_layout()
fig.subplots_adjust(hspace=0, wspace=0)

plt.savefig('../plots/Spec_logRMSE_row.pdf', dpi=300)
# plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# ------------------ INVERSE REDSHIFT INCOMPLETENESS ONLY ------------------
kind = 'invz'
stat = 'rmse'

fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(25, 7))

# ----------------- load the quantiles -----------------
name = 'Inverse Redshift Incompleteness Degradation'
degs = ['invz=0.1', 'invz=0.3', 'invz=0.6', 'invz=1.0', 'invz=1.4', 'LSSTerr_only']
colors = ['red', 'darkorange', 'goldenrod', 'green', 'blue', 'black']

t_dict  = getInvzQuants('TrainZ',  stat)
c_dict  = getInvzQuants('CMNN',    stat)
gl_dict = getInvzQuants('GPz_GL',  stat)
f_dict  = getInvzQuants('FZBoost', stat)
p_dict  = getInvzQuants('PZFlow',  stat)

algo_dicts = [t_dict, c_dict, gl_dict, f_dict, p_dict]
algo_names = ['TrainZ', 'CMNN', 'GPz GL', 'FZBoost', 'PZFlow']

# legend labels in correct order
legend_labels = ['$z_0=0.1$','$z_0=0.3$','$z_0=0.6$','$z_0=1.0$','$z_0=1.4$','LSST Error \n Model only']

# -------------- PLOTTING: 1 panel per algorithm --------------
for i, (dd, name_algo) in enumerate(zip(algo_dicts, algo_names)):
    ax = axes[i]

    cct = 0
    for key in dd:
        x = dd[key][0]

        # normalization logic taken from your big plot
        if name_algo == 'PZFlow':
            y = np.asarray(dd[key][1]) / np.max(dd[key][1])         # PZFlow normalized by max
        else:
            y = np.asarray(dd[key][1]) / 31247.0                    # others divided by 31247

        color = colors[cct]

        # TRAINZ gets legend labels, others do not
        if name_algo == 'TrainZ':
            ax.plot(x, y, color=color, label=legend_labels[cct])
        else:
            ax.plot(x, y, color=color)

        cct += 1

    # cosmetic tweaks common to all panels
    ax.set_title(name_algo, loc='center')
    ax.tick_params(axis='both')
    ax.set_xlim(-3.5, 4.1)      # RMSE range from your column 2
    if i != 0:
        ax.set_yticklabels([])

# only TrainZ panel has legend
axes[0].legend(fontsize=22, frameon=False)

fig.supxlabel('log(RMSE)')
fig.supylabel('Fraction of galaxies ≤ stat.', x=0.01)

fig.tight_layout()
fig.subplots_adjust(wspace=0.05)

plt.savefig('../plots/Invz_logRMSE_row.pdf', bbox_inches='tight', pad_inches=0)
# plt.show()


In [ ]:
makeBigQuantPlots('spec')

## GPz Log KLD plot, inv. redshift, $z_0 = 0.1$

In [ ]:
plt.hist(np.log(getStatsFile('GPz', 'kld')['LSSTerr_only']), density = True, bins = 100, alpha = 0.5, label = 'unbiased')
plt.hist(np.log(getStatsFile('GPz', 'kld')['invz=0.1']), density = True, bins = 100, alpha = 0.5, label = '$z_0=0.1$')
#plt.hist(np.log(getStatsFile('GPz', 'kld')['invz=0.6']), density = True, bins = 100, alpha = 0.5, label = '0.6')
#plt.hist(np.log(getStatsFile('GPz', 'kld')['invz=1.0']), density = True, bins = 100, alpha = 0.5, label = '1.0')
#plt.hist(np.log(getStatsFile('GPz', 'kld')['invz=1.4']), density = True, bins = 100, alpha = 0.5, label = '1.4')
plt.legend()
plt.xlabel('log KLD')
plt.ylabel('frequency')
plt.yscale('log')
plt.title("GPz outputs, training set: inverse redshift incompleteness, $z_0 = 0.1$")

In [ ]:
est = 'GPz_GL'
stat = 'ks'

#plt.hist(np.log(getStatsFile(est, stat)['LSSTerr_only']), density = True, bins = 100, alpha = 0.5, label = 'unbiased')
#plt.hist(np.log(getStatsFile(est, stat)['invz=0.1']), density = True, bins = 100, alpha = 0.5, label = '$z_0=0.1$')
#plt.hist(np.log(getStatsFile(est, stat)['invz=0.3']), density = True, bins = 100, alpha = 0.5, label = '$z_0=0.3$')
plt.hist(np.log(getStatsFile(est, stat)['invz=0.6']), density = True, bins = 100, alpha = 0.5, label = '$z_0=0.6$')
#plt.hist(np.log(getStatsFile(est, stat)['invz=1.0']), density = True, bins = 100, alpha = 0.5, label = '1.0')
plt.hist(np.log(getStatsFile(est, stat)['invz=1.4']), density = True, bins = 100, alpha = 0.5, label = '1.4')
plt.legend()
plt.xlabel('log KLD')
plt.ylabel('frequency')
plt.yscale('log')
plt.title(est+" outputs, training set: inverse redshift incompleteness, $z_0 = 0.1$")

# Point-to-distribution Statistics

## CDE Loss

In [ ]:
# Table of CDE Loss values 
pd.read_parquet('../paper_data/CDE_loss.pq')

In [ ]:
def CDEInvzScatter(est):
    df = pd.read_parquet('../paper_data/CDE_loss.pq')
    data = df[est]
    points = ax.plot([0.1, 0.3, 0.6, 1.0, 1.4], [data['invz=0.1'], data['invz=0.3'], data['invz=0.6'], data['invz=1.0'], data['invz=1.4']], alpha = 0.25)
    ax.scatter([0.1, 0.3, 0.6, 1.0, 1.4], [data['invz=0.1'], data['invz=0.3'], data['invz=0.6'], data['invz=1.0'], data['invz=1.4']], color = points[0].get_color(), label = est, )
    ax.plot(np.linspace(-1, 2, 3),data['LSSTerr_only']*np.ones(3), color = points[0].get_color(), alpha = 0.5, linestyle = 'dashed',)#label = 'no degradation' )
    #mpl.rcParams.update({'font.size':10})
    ax.tick_params(axis='both', size=7)
    #ax.set_xlabel('Pivot Redshift')
    #ax.set_ylabel('CDE Value')
    #ax.set_title('CDE Loss versus pivot redshift')

In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 14
plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)   # fontsize of the figure title
plt.rc('xtick', direction = 'in')
plt.rc('ytick', direction = 'in')
#cmap = sns.color_palette('colorblind')
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (7, 7))


CDEInvzScatter('TrainZ')#, ax)
CDEInvzScatter('CMNN')#, ax)
CDEInvzScatter('GPz_GL')#, ax)
CDEInvzScatter('FZBoost')#, ax)
CDEInvzScatter('PZFlow')#, ax)


plt.xlabel('Pivot Redshift')
plt.ylabel('CDE Loss Value')
#plt.title('CDE Loss versus pivot redshift', size = 'large')

plt.legend(prop = {'size': 10}, ncol = 2, loc = (0.55, 0.6))# loc = 'center right')#,fontsize='small')
plt.tight_layout()
plt.xlim(0, 1.6)
plt.ylim(-13.5, 2)
plt.savefig('../plots/Invz_CDE_Loss.pdf', dpi=300)

In [ ]:
spec_ls = ['BOSS', 'GAMA', 'zCOSMOS', 'DEEP2', 'VVDSf02', 'HSC']
    
cov_ls = []
for col in spec_ls: 
    cov_ls.append(np.sum(cov_df[col])/6)

print(cov_ls)

In [ ]:
cde_df = pd.read_parquet('../paper_data/CDE_loss.pq')
spec_cde_df = cde_df.drop(['invz=0.1', 'invz=0.3', 'invz=0.6', 'invz=1.0', 'invz=1.4', 'LSSTerr_only'] )
spec_cde_df

ests = ['TrainZ', 'CMNN', 'GPz_GL', 'FZBoost', 'PZFlow']



In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 18
plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)   # fontsize of the figure title
plt.rc('xtick', direction = 'in')
plt.rc('ytick', direction = 'in')
#cmap = sns.color_palette('colorblind')
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (16,8))#figsize = (30, 20))

pz_s_labels = ['2.5', '1.8', '100', '18.3', '48.6', '56.1']
pz_c_labels = ['2.5', '1.8', '18.3', '48.6', '56.1', '100']

for estim in ests:
    spec_ls = [spec_cde_df[estim]['BOSS'],
               spec_cde_df[estim]['GAMA'], 
               spec_cde_df[estim]['zCOSMOS'], 
               spec_cde_df[estim]['DEEP2'], 
               spec_cde_df[estim]['VVDSf02'], 
               spec_cde_df[estim]['HSC'] ]
    if estim == 'PZFlow': 
        for i, label in enumerate(pz_c_labels):    
            axes[0].scatter(cov_ls[i], spec_ls[i], s=550, edgecolor='tab:purple', facecolor='white', linewidth=2, zorder = 2)
            axes[0].text(cov_ls[i], spec_ls[i], label, fontsize=11, ha='center', va='center', color='purple', zorder = 3)
        axes[0].plot(cov_ls, spec_ls, color = 'tab:purple', zorder = 1)
    else: 
        print(len(cov_ls), len(spec_ls))
        axes[0].scatter(cov_ls, spec_ls)
        axes[0].plot(cov_ls, spec_ls)
    axes[0].set_xlabel('Percentage of test set covered')#, fontsize = 30)
    
ct = 0 
for estim in ests: 
    gal_nums = [195, 2099, 14253, 32089, 38385, 80845]
    nums_ls = [ spec_cde_df[estim]['BOSS'],
                spec_cde_df[estim]['GAMA'], 
                spec_cde_df[estim]['HSC'], 
                spec_cde_df[estim]['zCOSMOS'], 
                spec_cde_df[estim]['DEEP2'], 
                spec_cde_df[estim]['VVDSf02'] ]
    
    if estim == 'PZFlow': 
        for i, label in enumerate(pz_s_labels):    
            axes[1].scatter(gal_nums[i], nums_ls[i], s=550, edgecolor='tab:purple', facecolor='white', linewidth=2, zorder = 2)
            axes[1].text(gal_nums[i], nums_ls[i], label, fontsize=11, ha='center', va='center', color='purple', zorder = 3)
        axes[1].plot(gal_nums, nums_ls, color = 'tab:purple', zorder = 1)
    else: 
        axes[1].plot(gal_nums, nums_ls, label = estim)
        axes[1].scatter(gal_nums, nums_ls)
    

    axes[1].semilogx()
    axes[1].set_xlabel('Number of training galaxies')#, fontsize = 30)
    axes[1].legend()

    if ct == 3: 
        axes[1].plot([], [], color = "tab:purple", label = 'PZFlow')
    ct += 1

axes[0].set_ylabel("CDE Loss", x=-0.1)

axes[0].set_title("CDE Loss vs. Average Coverage", y=1.02, size = 'large')
axes[1].set_title("CDE Loss vs. Training Set Size", y=1.02, size = 'large')

#fig.suptitle("CDE Loss as function of training set coverage and size")
#fig.tight_layout
fig.subplots_adjust(hspace = 0.05, wspace = 0.1)




In [ ]:
import matplotlib.pyplot as plt

# Sample data
x = [1, 2, 3, 4, 5]
y = [5, 3, 8, 6, 7]
labels = ['A', 'B', 'C', 'D', 'E']

# Create the scatter plot
plt.figure(figsize=(8, 6))
for i, label in enumerate(labels):
    # Plot each point as a large, unfilled purple circle
    plt.scatter(x[i], y[i], s=500, edgecolor='purple', facecolor='none', linewidth=2)
    # Add the label in the center of each circle, also in purple
    plt.text(x[i], y[i], label, fontsize=12, ha='center', va='center', color='purple')

# Adding titles and labels
plt.title('Scatterplot with Purple Circles and Labels', fontsize=14)
plt.xlabel('X-axis', fontsize=12)
plt.ylabel('Y-axis', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
spec_rej_df = pd.read_parquet('../paper_data/3sigma_spec_outliers.pq')

ests = ['TrainZ', 'CMNN', 'GPz_GL', 'FZBoost', 'PZFlow']
float(spec_rej_df['TrainZ'][0][0:-1])

spec_rej_df
#float(spec_rej_df[estim]['BOSS'][0:-1])

In [ ]:
fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (20, 10))#figsize = (30, 20))

for estim in ests:
    spec_ls = [float(spec_rej_df[estim][0][0:-1]),
               float(spec_rej_df[estim][2][0:-1]),
               float(spec_rej_df[estim][5][0:-1]), 
               float(spec_rej_df[estim][1][0:-1]),  
               float(spec_rej_df[estim][4][0:-1]), 
               float(spec_rej_df[estim][3][0:-1]),  
               ]
    axes[0].scatter(cov_ls, spec_ls)
    axes[0].plot(cov_ls, spec_ls)
    axes[0].set_xlabel('Percentage of test set covered')
    

for estim in ests: 
    gal_nums = [195, 2099, 14253, 32089, 38385, 80845]
    nums_ls = [float(spec_rej_df[estim][0][0:-1]),
               float(spec_rej_df[estim][2][0:-1]),
               float(spec_rej_df[estim][3][0:-1]), 
               float(spec_rej_df[estim][5][0:-1]),  
               float(spec_rej_df[estim][1][0:-1]), 
               float(spec_rej_df[estim][4][0:-1]),  
               ]

    axes[1].plot(gal_nums, nums_ls, label = estim)
    axes[1].scatter(gal_nums, nums_ls)
    axes[1].semilogx()
    axes[1].set_xlabel('Number of training galaxies')
    axes[1].legend()

fig.supylabel('Outlier rate', x=0.07)
fig.suptitle("Outlier rate as function of training set coverage and size")
fig.tight_layout


## PIT

In [ ]:
def getPITFile(est):
    return pd.read_parquet('../paper_data/'+est+'/PIT.pq')

In [ ]:
est = 'GPz_GL'
# I've put the y-axis on a log scale; otherwise the spikes on the sides are too large to see the rest of the distribution

In [ ]:
# plt.hist(getPITFile(est)['LSSTerr_only'], bins = 100, alpha=0.2, label = 'undegraded', density=True)#, density = True)
# plt.hist(removeNans(getPITFile(est), ['invz=0.1'])['invz=0.1'], bins = 100, alpha=0.2, label = '$z_0 = 0.1$', density=True)
# plt.hist(removeNans(getPITFile(est), ['invz=0.3'])['invz=0.3'], bins = 100, alpha=0.2, label = '$z_0 = 0.3$', density=True)
# plt.hist(removeNans(getPITFile(est), ['invz=0.6'])['invz=0.6'], bins = 100, alpha=0.2, label = '$z_0 = 0.6$', density=True)
# plt.hist(removeNans(getPITFile(est), ['invz=1.0'])['invz=1.0'], bins = 100, alpha=0.2, label = '$z_0 = 1.0$', density=True)
# plt.hist(removeNans(getPITFile(est), ['invz=1.4'])['invz=1.4'], bins = 100, alpha=0.2, label = '$z_0 = 1.4$', density=True)

# x = np.linspace(0, 1, 100)
# #plt.plot(x, sts.uniform.pdf(x), label = 'Uniform Distribution')
# plt.yscale('log')
# plt.legend()
# plt.xlabel("PIT Value")
# plt.ylabel('Log Bin Count')
# plt.title('Probability Integral Transform Distribution for '+est+' with Inverse Redshift Incompleteness Training Data')

In [ ]:
# plt.hist(getPITFile(est)['LSSTerr_only'], bins = 100, alpha=0.5, label = 'undegraded')#, density = True)
# plt.hist(removeNans(getPITFile(est), ['BOSS'])['BOSS'], bins = 100, alpha=0.5, label = 'BOSS')
# #plt.hist(removeNans(getPITFile(est), ['DEEP2'])['DEEP2'], bins = 100, alpha=0.25, label = 'DEEP2')
# #plt.hist(removeNans(getPITFile(est), ['GAMA'])['GAMA'], bins = 100, alpha=0.25, label = 'GAMA')
# #plt.hist(removeNans(getPITFile(est), ['HSC'])['HSC'], bins = 100, alpha=0.25, label = 'HSC')
# #plt.hist(removeNans(getPITFile(est), ['VVDSf02'])['VVDSf02'], bins = 100, alpha=0.25, label = 'VVDSf02')
# #plt.hist(removeNans(getPITFile(est), ['zCOSMOS'])['zCOSMOS'], bins = 100, alpha=0.25, label = 'zCOSMOS')

# x = np.linspace(0, 1, 100)
# plt.plot(x, sts.uniform.pdf(x), label = 'Uniform Distribution')
# #plt.yscale('log')
# plt.legend()
# plt.xlabel("PIT Value")
# plt.ylabel('Log Bin Count')
# plt.title('Probability Integral Transform Distribution for '+est+' with Spectroscopic Survey Training Data')

In [ ]:
est_ls = ['TrainZ', 'CMNN', 'GPz_GL', 'FZBoost', 'PZFlow']
binnum = 50

fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(20, 30))#, figsize=(8, 40))

ct = 0
for est in est_ls:
    one = np.histogram(removeNans(getPITFile(est), ['invz=0.1'])['invz=0.1'], bins = binnum, density = True)
    three = np.histogram(removeNans(getPITFile(est), ['invz=0.3'])['invz=0.3'], bins = binnum, density = True)
    six = np.histogram(removeNans(getPITFile(est), ['invz=0.6'])['invz=0.6'], bins = binnum, density = True)
    one_zero = np.histogram(removeNans(getPITFile(est), ['invz=1.0'])['invz=1.0'], bins = binnum, density = True)
    one_four = np.histogram(removeNans(getPITFile(est), ['invz=1.4'])['invz=1.4'], bins = binnum, density = True)
    control = np.histogram(removeNans(getPITFile(est), ['LSSTerr_only'])['LSSTerr_only'], bins = binnum, density = True)

    grid = np.delete(one[1], -1) + (one[1][1] - one[1][0])/2

# colors = ['red', 'darkorange', 'goldenrod', 'green', 'blue', 'black']

    if est == 'GPz_GL': 
        axes[ct][0].plot(grid, one[0], color = 'red')
        axes[ct][0].plot(grid, three[0], color = 'darkorange')
        axes[ct][0].plot(grid, six[0], color = 'goldenrod')
        axes[ct][0].plot(grid, one_zero[0], color = 'green')
        axes[ct][0].plot(grid, one_four[0], color = 'blue')
        axes[ct][0].plot(grid, control[0], color = 'brown')
        axes[ct][0].plot(grid, np.ones(len(grid)), color = 'k', linestyle='solid')
        #axes[ct][0].set_yticklabels([])
        #axes[ct].semilogy()
        #axes[ct].legend(prop = {'size': 10} )

    elif est == 'TrainZ': 
        axes[ct][0].plot(grid, one[0], label = '$z_0=0.1$', color = 'red')
        axes[ct][0].plot(grid, three[0], label = '$z_0=0.3$', color = 'darkorange')
        axes[ct][0].plot(grid, six[0], label = '$z_0=0.6$', color = 'goldenrod')
        axes[ct][0].plot(grid, one_zero[0], label = '$z_0=1.0$', color = 'green')
        axes[ct][0].plot(grid, one_four[0], label = '$z_0=1.4$', color = 'blue')
        axes[ct][0].plot(grid, control[0], label = 'LSST error only', color = 'brown')
        axes[ct][0].plot(grid, np.ones(len(grid)), color = 'k', linestyle='solid')
        #axes[ct][0].set_yticklabels([])
        axes[ct][0].legend(prop = {'size': 15})


    else: 
        axes[ct][0].plot(grid, one[0])
        axes[ct][0].plot(grid, three[0])
        axes[ct][0].plot(grid, six[0])
        axes[ct][0].plot(grid, one_zero[0])
        axes[ct][0].plot(grid, one_four[0])
        axes[ct][0].plot(grid, control[0])
        axes[ct][0].plot(grid, np.ones(len(grid)), color = 'k', linestyle='solid')
        #if ct != 4: 
            #axes[ct][0].set_yticklabels([])
        #axes[ct].semilogy()
        #axes[ct].legend()

    axes[ct][0].set_ylim(0, 4.5)
    ct += 1

mpl.rcParams.update({'font.size':20})
axes[0][0].set_title('TrainZ', size = 'large', y=0.85)
axes[1][0].set_title('CMNN',size = 'large', y=0.85)
axes[2][0].set_title('GPz GL',size = 'large', y=0.85)
axes[3][0].set_title('FlexZBoost',size = 'large', y=0.85)
axes[4][0].set_title('PzFlow',size = 'large', y=0.85)


axes[0][0].set_title("Inverse Redshift Incompleteness", size = 'x-large', y=1.05)




ct =0 

for est in est_ls:
    BOSS = np.histogram(removeNans(getPITFile(est), ['BOSS'])['BOSS'], bins = binnum, density = True)
    DEEP2 = np.histogram(removeNans(getPITFile(est), ['DEEP2'])['DEEP2'], bins = binnum, density = True)
    GAMA = np.histogram(removeNans(getPITFile(est), ['GAMA'])['GAMA'], bins = binnum, density = True)
    HSC = np.histogram(removeNans(getPITFile(est), ['HSC'])['HSC'], bins = binnum, density = True)
    VVDSf02 = np.histogram(removeNans(getPITFile(est), ['VVDSf02'])['VVDSf02'], bins = binnum, density = True)
    zCOSMOS = np.histogram(removeNans(getPITFile(est), ['zCOSMOS'])['zCOSMOS'], bins = binnum, density = True)
    control = np.histogram(removeNans(getPITFile(est), ['LSSTerr_only'])['LSSTerr_only'], bins = binnum, density = True)

    grid = np.delete(BOSS[1], -1) + (BOSS[1][1] - BOSS[1][0])/2

    if ct == 0:
        axes[ct][1].plot(grid, BOSS[0], label = 'pseudo-BOSS')
        axes[ct][1].plot(grid, GAMA[0], label = 'GAMA')
        axes[ct][1].plot(grid, HSC[0], label = 'HSC')
        axes[ct][1].plot(grid, zCOSMOS[0], label = 'zCOSMOS')
        axes[ct][1].plot(grid, DEEP2[0], label = 'pseudo-DEEP2')
        axes[ct][1].plot(grid, VVDSf02[0], label = 'VVDSf02')
        axes[ct][1].plot(grid, control[0], label = 'LSST error only')
        axes[ct][1].plot(grid, np.ones(len(grid)), color = 'k', linestyle='solid')
        axes[ct][1].semilogy()
        axes[ct][1].set_ylim(10e-4,10e2)
        #axes[ct][1].set_yticklabels([])
        axes[ct][1].legend(prop = {'size': 15}, loc = 'upper left', ncols = 2)

    else: 
        axes[ct][1].plot(grid, BOSS[0])
        axes[ct][1].plot(grid, DEEP2[0])
        axes[ct][1].plot(grid, GAMA[0])
        axes[ct][1].plot(grid, HSC[0])
        axes[ct][1].plot(grid, VVDSf02[0])
        axes[ct][1].plot(grid, zCOSMOS[0])
        axes[ct][1].plot(grid, control[0])
        axes[ct][1].plot(grid, np.ones(len(grid)), color = 'k', linestyle='solid')
        axes[ct][1].semilogy()
        axes[ct][1].set_ylim(10e-4,10e2)
        # if ct != 4: 
        #     axes[ct][1].set_yticklabels([])

    ct += 1

mpl.rcParams.update({'font.size':20})

axes[0][1].set_title('TrainZ', size = 'large', y=0.85)
axes[1][1].set_title('CMNN',size = 'large', y=0.85)
axes[2][1].set_title('GPz GL',size = 'large', y=0.85)
axes[3][1].set_title('FlexZBoost',size = 'large',y=0.85)
axes[4][1].set_title('PzFlow',size = 'large', y=0.85)


axes[4][0].set_xlabel('PIT Value', size = 'x-large')
axes[4][1].set_xlabel('PIT Value', size = 'x-large')

axes[0][1].set_title("Spectroscopic Survey Degradation", size = 'x-large', y=1.05)

axes[0][0].set_ylabel('Normalized Bin Count', x=1)
axes[1][0].set_ylabel('Normalized Bin Count', x=-0.2)
axes[2][0].set_ylabel('Normalized Bin Count', x=-0.2)
axes[3][0].set_ylabel('Normalized Bin Count', x=-0.2)
axes[4][0].set_ylabel('Normalized Bin Count', x=-0.2)

#fig.tight_layout()
fig.subplots_adjust(hspace = 0.15, wspace = 0.15)


#plt.savefig('Invz_PIT.pdf', bbox_inches = 'tight', pad_inches = 0)
    

In [ ]:
# est_ls = ['TrainZ', 'CMNN', 'GPz_GL', 'FZBoost', 'PZFlow']
# binnum = 50

# fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(10, 20))

# ct =0 

# for est in est_ls:
#     BOSS = np.histogram(removeNans(getPITFile(est), ['BOSS'])['BOSS'], bins = binnum, density = True)
#     DEEP2 = np.histogram(removeNans(getPITFile(est), ['DEEP2'])['DEEP2'], bins = binnum, density = True)
#     GAMA = np.histogram(removeNans(getPITFile(est), ['GAMA'])['GAMA'], bins = binnum, density = True)
#     HSC = np.histogram(removeNans(getPITFile(est), ['HSC'])['HSC'], bins = binnum, density = True)
#     VVDSf02 = np.histogram(removeNans(getPITFile(est), ['VVDSf02'])['VVDSf02'], bins = binnum, density = True)
#     zCOSMOS = np.histogram(removeNans(getPITFile(est), ['zCOSMOS'])['zCOSMOS'], bins = binnum, density = True)
#     control = np.histogram(removeNans(getPITFile(est), ['LSSTerr_only'])['LSSTerr_only'], bins = binnum, density = True)

#     grid = np.delete(BOSS[1], -1) + (BOSS[1][1] - BOSS[1][0])/2

#     if ct == 0:
#         axes[ct].plot(grid, BOSS[0], label = 'pseudo-BOSS')
#         axes[ct].plot(grid, GAMA[0], label = 'GAMA')
#         axes[ct].plot(grid, HSC[0], label = 'HSC')
#         axes[ct].plot(grid, zCOSMOS[0], label = 'zCOSMOS')
#         axes[ct].plot(grid, DEEP2[0], label = 'pseudo-DEEP2')
#         axes[ct].plot(grid, VVDSf02[0], label = 'VVDSf02')
#         axes[ct].plot(grid, control[0], label = 'LSST error only')
#         axes[ct].plot(grid, np.ones(len(grid)), color = 'k', linestyle='solid')
#         axes[ct].semilogy()
#         axes[ct].set_ylim(10e-4,10e2)
#         axes[ct].set_yticklabels([])
#         axes[ct].legend(prop = {'size': 10}, loc = 'upper left' )

#     else: 
#         axes[ct].plot(grid, BOSS[0])
#         axes[ct].plot(grid, DEEP2[0])
#         axes[ct].plot(grid, GAMA[0])
#         axes[ct].plot(grid, HSC[0])
#         axes[ct].plot(grid, VVDSf02[0])
#         axes[ct].plot(grid, zCOSMOS[0])
#         axes[ct].plot(grid, control[0])
#         axes[ct].plot(grid, np.ones(len(grid)), color = 'k', linestyle='solid')
#         axes[ct].semilogy()
#         axes[ct].set_ylim(10e-4,10e2)
#         if ct != 4: 
#             axes[ct].set_yticklabels([])

#     ct += 1

# mpl.rcParams.update({'font.size':20})

# axes[0].set_title('TrainZ', size = 'large', y=0.85)
# axes[1].set_title('CMNN',size = 'large', y=0.85)
# axes[2].set_title('GPz GL',size = 'large', y=0.85)
# axes[3].set_title('FlexZBoost',size = 'large',y=0.85)
# axes[4].set_title('PzFlow',size = 'large', y=0.85)

# fig.supylabel('Normalized Bin Count', size = 'x-large', x = -0.01)
# fig.supxlabel('PIT Value', size = 'x-large')
# fig.suptitle('Spectroscopic Survey PIT Distributions', size = 'xx-large')
# #fig.legend(prop={'size': 12}, loc = 'upper left')


# fig.tight_layout()
# fig.subplots_adjust(hspace = 0, wspace = 0)

# #plt.savefig('Spec_PIT.pdf', bbox_inches = 'tight', pad_inches = 0)


# Mean Point Estimates

In [ ]:
import pandas as pd

outs_df = pd.read_parquet('../paper_data/3sigma_invz_outliers.pq')
outs_df
# Outlier Rejection statistics on mode data

In [ ]:
# data

# float(outs_df['CMNN']['LSSTerr_only'][0:-1])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 18
plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)   # fontsize of the figure title
plt.rc('xtick', direction = 'in')
plt.rc('ytick', direction = 'in')
#cmap = sns.color_palette('colorblind')
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
est_ls = ['TrainZ', 'CMNN', 'GPz_GL', 'FZBoost', 'PZFlow']

fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (15, 7))



#fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (7, 7))
CDEInvzScatter('TrainZ', ax[0])
CDEInvzScatter('CMNN', ax[0])
CDEInvzScatter('GPz_GL', ax[0])
CDEInvzScatter('FZBoost', ax[0])
CDEInvzScatter('PZFlow', ax[0])


ax[0].set_xlabel('Pivot redshift')
ax[0].set_ylabel('CDE Loss Value')
#plt.title('CDE Loss versus pivot redshift', size = 'large')

ax[0].legend(prop = {'size': 15}, ncol = 2)#, loc = (0.55, 0.6))# loc = 'center right')#,fontsize='small')
ax[0].set_xlim(0, 1.6)
ax[0].set_ylim(-13.5, 3.5)
#ax[0].savefig('../plots/Invz_CDE_Loss.pdf', dpi=300)



for est in est_ls: 
    col = outs_df[est]
    y = [float(col[0][0:-1]), float(col[1][0:-1]), float(col[2][0:-1]), float(col[3][0:-1]), float(col[4][0:-1])]
    x = [0.1, 0.3, 0.6, 1.0, 1.4]
    lin = [-2, -1, 0, 1, 2]
    ax[1].scatter(x, y, label = est)
    points = ax[1].plot(x, y, alpha=0.5)
    if est == 'FZBoost':
        ax[1].plot(lin, float(col[5][0:-1])*np.ones(5), color = points[0].get_color(), linestyle='dashed', alpha=0.5)
    else:
        ax[1].plot(lin, float(col[5][0:-1])*np.ones(5), color = points[0].get_color(), linestyle='dashed', alpha=0.5)
ax[1].legend(ncols = 2, prop = {'size': 15} )
ax[1].set_xlabel('Pivot redshift')
ax[1].set_ylabel('Catastrophic outlier rate (%)')
#plt.title('Outlier Rejection catastrophic outlier rate as a function of pivot redshift')
ax[1].set_ylim(3, 33)
ax[1].set_xlim(0, 1.5)

ax[0].set_title("CDE Loss", size = 'x-large', y=1.02)
ax[1].set_title("Outlier Rejection", size = 'x-large', y=1.02)
#ax[1].savefig('../plots/Invz_Out_Rej.pdf', dpi=300)

In [ ]:
def getModesFile(est):
    return pd.read_parquet('./'+est+'/modes.pq')

In [ ]:
# file = getModesFile('GPz_GL')
# file.insert(0, 'redshift', getModesFile('GPz')['redshift'])
# file.to_parquet('../paper_data/GPz_GL/modes.pq')
# file

In [ ]:
def getStdev(est, deg):
    file = pd.read_parquet('../paper_data/outlier_rejection_mean_stdev.pq')
    return file[est][deg][1]

In [ ]:
# choose which training set degradation condition you're plotting by setting deg = [degrader]
# options are in deg_ls

deg = 'LSSTerr_only'

In [ ]:
pd.read_parquet('../paper_data/3sigma_spec_outliers.pq')



In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 30

plt.rc('font', size=BIGGER_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=BIGGER_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=BIGGER_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)   # fontsize of the figure title
plt.rc('xtick', direction = 'in')
plt.rc('ytick', direction = 'in')
#cmap = sns.color_palette('colorblind')
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import pandas as pd

fig, axes = plt.subplots(nrows=2, ncols=2, figsize = (20, 20) )

line = np.asarray(np.linspace(0, 3, 101))

xmax = 2.99
ymin = 0

mpl.rcParams.update({'font.size':30})

m = axes[0][0].hist2d(getModesFile('CMNN')['redshift'], getModesFile('CMNN')[deg], bins = 100, norm = mpl.colors.LogNorm())
axes[0][0].plot(line, line, color = 'orange', linestyle = 'dashed', linewidth = 3)
axes[0][0].plot(line, line + 5*getStdev('CMNN', deg)*(1+line), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[0][0].plot(line, line - 5*getStdev('CMNN', deg)*(1+line), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[0][0].set_title('CMNN', y=0.95)
axes[0][0].set_xlim(0, xmax)
axes[0][0].set_ylim(ymin, 3)
axes[0][0].set_xticklabels([])
axes[0][0].set_ylabel('Mode photo-z')
axes[0][0].tick_params()
axes[0][0].plot([], [], label = "8.016%", color = 'red', linestyle = 'dashed')
axes[0][0].legend()

n = axes[0][1].hist2d(getModesFile('GPz_GL')['redshift'], getModesFile('GPz_GL')[deg], bins = 100, norm = mpl.colors.LogNorm())
axes[0][1].plot(line, line, color = 'orange', linestyle = 'dashed', linewidth = 3)
axes[0][1].plot(line, line + 5*getStdev('GPz_GL', deg)*(1+line), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[0][1].plot(line, line - 5*getStdev('GPz_GL', deg)*(1+line), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[0][1].set_title('GPz GL', y=0.95)
axes[0][1].set_xlim(0, xmax)
axes[0][1].set_ylim(ymin, 3)
axes[0][1].set_xticklabels([])
axes[0][1].set_yticklabels([])
axes[0][1].tick_params()
axes[0][1].plot([], [], label = "15.81%", color = 'red', linestyle = 'dashed')
axes[0][1].legend()

h = axes[1][0].hist2d(getModesFile('FZBoost')['redshift'], getModesFile('FZBoost')[deg], bins = 100, norm = mpl.colors.LogNorm())
axes[1][0].plot(line, line, color = 'orange', linestyle = 'dashed', linewidth = 3)
axes[1][0].plot(line, line + 5*(1+line)*getStdev('FZBoost', deg), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[1][0].plot(line, line - 5*(1+line)*getStdev('FZBoost', deg), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[1][0].set_title('FZBoost', y=0.95)
axes[1][0].set_xlim(0, xmax)
axes[1][0].set_xlabel('Spec z')
axes[1][0].set_ylim(ymin, 2.99)
axes[1][0].set_ylabel('Mode photo-z')
axes[1][0].tick_params()
axes[1][0].plot([], [], label = "7.667%", color = 'red', linestyle = 'dashed')
axes[1][0].legend()

inds = []
z = []
data = []
pzfdata = getModesFile('PZFlow')[deg].tolist()
ct = 0
for i in pzfdata:
    if np.isnan(i)==False: 
        data.append(i)
        inds.append(ct)
    ct += 1

ct2 = 0
for j in getModesFile('PZFlow')['redshift']:
    if ct2 in inds: 
        z.append(j)
    ct2 += 1

axes[1][1].hist2d(z, data, bins = 100, norm = mpl.colors.LogNorm())
axes[1][1].plot(line, line, color = 'orange', linestyle = 'dashed', linewidth = 3)
axes[1][1].plot(line, line + 5*(1+line)*getStdev('PZFlow', deg), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[1][1].plot(line, line - 5*(1+line)*getStdev('PZFlow', deg), color = 'red', linestyle = 'dashed', linewidth = 3)
axes[1][1].set_title('PZFlow', y=0.95)
axes[1][1].set_xlim(0, 3.0)
axes[1][1].set_ylim(ymin, 2.99)
#axes[1][1].set_xticklabels([])
axes[1][1].set_yticklabels([])
axes[1][1].set_xlabel('Spec z')
axes[1][1].tick_params()
axes[1][1].plot([], [], label = "8.699%", color = 'red', linestyle = 'dashed')
axes[1][1].legend()


#fig.suptitle("Mode point estimates for LSST error model only training set degradation", size = 'x-large')
#fig.supxlabel('Spectroscopic redshift')
#fig.supylabel('Mode of photometric redshift')


cbar_ax = fig.add_axes([ 1.0, 0.1, 0.03, 0.75])
cbar = fig.colorbar(n[3], cax=cbar_ax)
cbar.set_label(label = 'data density', size = 'large' )
cbar.ax.tick_params(labelsize = 40)
#fig.legend()


#fig.colorbar(n[3], ax=axes, label = 'data density')

fig.tight_layout()
fig.subplots_adjust(hspace = 0.05, wspace = 0.05)
#plt.savefig('Control_Modes.pdf', bbox_inches = 'tight', pad_inches = 0)

plt.savefig('../plots/Control_Out_Rej.pdf', dpi=300)



# Failiure Rates

In [ ]:
getStatsFile('PZFlow', 'kld')

In [ ]:
ls = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

for i in ls:
    file = gethdf5('GPz', 'LSSTerr_only', i)
    print(file[1])

In [ ]:
# ens = qp.read(getEnsFilepath('GPz', 'LSSTerr_only'))
# h = h5py.File(getEnsFilepath('GPz', 'LSSTerr_only'))
# grid = np.linspace(0, 3, 301)
# print(ens[0].pdf(grid))
# print(h['data']['loc'][0], h['data']['scale'][0] ) 

In [ ]:
plt.plot(grid, ens)

In [ ]:
pd.read_parquet('../paper_data/data/rubin_roman_catalog.pq')

In [ ]:
pd.read_parquet('../paper_data/data/trimmed_rubin_roman_catalog.pq')

In [ ]:
pd.read_parquet('../paper_data/data/1e6_trimmed_rubin_roman_catalog.pq')

# Other

In [ ]:
import scipy.stats as sts
import sklearn.metrics as skl
import scipy.special as sps

base_dist = sts.norm(loc=1, scale=0.08)
base_wide_dist = sts.norm(loc=1, scale=0.4)
wide_dist = sts.norm(loc=1.1, scale=0.4)
narrow_dist = sts.norm(loc=1.25, scale=0.08)

grid = np.linspace(0, 3, 30000)


In [ ]:
plt.plot(grid, base_dist.cdf(grid))
plt.show()

In [ ]:
def getCDELoss(dist,z_true):
    import scipy.integrate

    square_ls = []
    pdf_ls = []


    arr = dist.pdf(grid)**2
    square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.01))
#     print(" square integration done")

    pdf_ls.append(dist.pdf(z_true))
#     print(" pdf evaluation done")
    
#     print(square_ls, pdf_ls)
    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    print( str(term_1 - 2*term_2))
    
    return term_1 - 2*term_2

In [ ]:
samples = base_dist.rvs(100)
plt.hist(samples)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

nsamples = 1000000
base_samples = base_dist.rvs(nsamples)

mse_self = mean_squared_error(base_samples, base_samples)
mse_self

In [ ]:
0.08**2*2

In [ ]:
nsamples = 1000000
base_samples = base_dist.rvs(nsamples)

bcvm = sts.cramervonmises(base_samples, base_dist.cdf).statistic/nsamples
bwcvm = sts.cramervonmises(base_samples, base_wide_dist.cdf).statistic/nsamples
ncvm = sts.cramervonmises(base_samples, narrow_dist.cdf).statistic/nsamples
wcvm = sts.cramervonmises(base_samples, wide_dist.cdf).statistic/nsamples

bks = sts.kstest(base_samples, base_dist.cdf).statistic
bwks = sts.kstest(base_samples, base_wide_dist.cdf).statistic
nks = sts.kstest(base_samples, narrow_dist.cdf).statistic
wks = sts.kstest(base_samples, wide_dist.cdf).statistic

# bks = sts.kstest(base_samples, base_dist.rvs(nsamples)).statistic
# bwks = sts.kstest(base_samples, base_wide_dist.rvs(nsamples)).statistic
# nks = sts.kstest(base_samples, narrow_dist.rvs(nsamples)).statistic
# wks = sts.kstest(base_samples, wide_dist.rvs(nsamples)).statistic

brmse = skl.mean_squared_error(base_samples, base_dist.rvs(nsamples))
bwrmse = skl.mean_squared_error(base_samples, base_wide_dist.rvs(nsamples))
nrmse = skl.mean_squared_error(base_samples, narrow_dist.rvs(nsamples))
wrmse = skl.mean_squared_error(base_samples, wide_dist.rvs(nsamples))

bdata = base_dist.pdf(grid)
bwdata = base_wide_dist.pdf(grid)
wdata = wide_dist.pdf(grid)
ndata = narrow_dist.pdf(grid)

bdata[bdata < 1e-300] = 1e-300
bwdata[bwdata < 1e-300] = 1e-300
wdata[wdata < 1e-300] = 1e-300
ndata[ndata < 1e-300] = 1e-300

bkld = sum(sps.kl_div(bdata, bdata))*3/30000
bwkld = sum(sps.kl_div(bdata, bwdata))*3/30000
nkld = sum(sps.kl_div(bdata, ndata))*3/30000
wkld = sum(sps.kl_div(bdata, wdata))*3/30000

bwdist = sts.wasserstein_distance(base_samples, base_dist.rvs(nsamples))
bwwdist = sts.wasserstein_distance(base_samples, base_wide_dist.rvs(nsamples)) 
nwdist = sts.wasserstein_distance(base_samples, narrow_dist.rvs(nsamples))
wwdist = sts.wasserstein_distance(base_samples, wide_dist.rvs(nsamples))

bcdf = base_dist.cdf(1)
bwcdf = base_wide_dist.cdf(1)
ncdf = narrow_dist.cdf(1)
wcdf = wide_dist.cdf(1)

bcde = getCDELoss(base_dist, 1.0)
bwcde = getCDELoss(base_wide_dist, 1.0)
ncde = getCDELoss(narrow_dist, 1.0)
wcde = getCDELoss(wide_dist, 1.0)

print(ncdf)

In [ ]:
str(nks)

In [ ]:
plt.rcParams['text.usetex'] = True



In [ ]:
fig, axes = plt.subplots(nrows = 2, ncols= 2, figsize = (16, 16))

axes[0][0].plot(grid, base_dist.pdf(grid), color = 'k')
axes[0][0].plot(np.ones(21), np.linspace(0, 20, 21), color = 'k', linestyle='dashed')

axes[0][0].scatter([], [], color = 'white', label = "CvM: " + str(round(bcvm,3)))
axes[0][0].scatter([], [], color = 'white', label = "KS: " + str(round(bks,2)))
axes[0][0].scatter([], [], color = 'white', label = "RMSE: " + str(round(brmse, 3)))
axes[0][0].scatter([], [], color = 'white', label = "KLD: " + str(round(bkld, 3)))
axes[0][0].scatter([], [], color = 'white', label = "W Dist: " + str(round(bwdist, 2)))
axes[0][0].scatter([], [], color = 'white', label = "CDE Loss: " + str(round(bcde, 3)))

# axes[0][0].text(
#     0.65, 0.95,
#     f"CvM: {float(bcvm):.4f}\n"
#     f"KS: {float(bks):.4f}\n"
#     f"RMSE: {float(brmse):.4f}\n"
#     f"KLD: {float(bkld):.4f}\n"
#     f"W Dist: {float(bwdist):.4f}\n"
#     f"CDF: {float(bcdf):.4f}\n"
#     f"CDE Loss: {float(bcde):.4f}",
#     transform=axes[0][0].transAxes,
#     ha='left', va='top',
#     fontsize=18,
#     bbox=dict(facecolor='white', alpha=0.85, edgecolor='black')
# )

axes[0][0].legend(fontsize = 18, loc = 'upper right')
axes[0][0].set_ylim(0, 6)
axes[0][0].set_xlim(0, 2.5)

axes[0][0].set_xticklabels([])
axes[0][0].set_ylabel('$P(z|photometry)$')
axes[0][0].set_title('precise, \n accurate', x=0.15, y=0.85)

# axes[0][0].scatter([], [], color='white',
#                    label=r"\textbf{\textcolor{red}{CvM: " + str(bcvm)[0:5] + "}}")
# axes[0][0].legend(fontsize=18, loc='upper right', handlelength=0)


axes[0][1].plot(grid, base_dist.pdf(grid), color = 'k')
axes[0][1].plot(np.ones(21), np.linspace(0, 20, 21), color = 'k',)
axes[0][1].plot(grid, base_wide_dist.pdf(grid), color = 'dodgerblue')
axes[0][1].plot(np.ones(21), np.linspace(0, 20, 21), color = 'dodgerblue', linestyle='dashed')

axes[0][1].scatter([], [], color = 'white', label = "CvM: " + str(bwcvm)[0:4])
axes[0][1].scatter([], [], color = 'white', label = "KS: " + str(bwks)[0:4])
axes[0][1].scatter([], [], color = 'white', label = "RMSE: " + str(bwrmse)[0:4])
axes[0][1].scatter([], [], color = 'white', label = "KLD: " + str(bwkld)[0:4])
axes[0][1].scatter([], [], color = 'white', label = "W Dist: " + str(bwwdist)[0:4])
axes[0][1].scatter([], [], color = 'white', label = "CDE Loss: " + str(bwcde)[0:4])
axes[0][1].legend(fontsize = 18, loc = 'upper right')
axes[0][1].set_ylim(0, 6)
axes[0][1].set_xlim(0, 2.5)

# axes[0][1].set_xticklabels([])
axes[0][1].set_yticklabels([])
axes[0][1].set_title(' not precise, \n accurate', x=0.2, y=0.85)
axes[0][1].set_xlabel('Redshift')



axes[1][0].plot(grid, base_dist.pdf(grid), color = 'k')
axes[1][0].plot(np.ones(21), np.linspace(0, 20, 21), color = 'k', linestyle='dashed')
axes[1][0].plot(grid, narrow_dist.pdf(grid), color = 'orange')
axes[1][0].plot(1.25*np.ones(21), np.linspace(0, 20, 21), color = 'orange', linestyle='dashed')

axes[1][0].scatter([], [], color = 'white', label = "CvM: " + str(ncvm)[0:4])
axes[1][0].scatter([], [], color = 'white', label = "KS: " + str(nks)[0:4])
axes[1][0].scatter([], [], color = 'white', label = "RMSE: " + str(nrmse)[0:4])
axes[1][0].scatter([], [], color = 'white', label = "KLD: " + str(nkld)[0:4])
axes[1][0].scatter([], [], color = 'white', label = "W Dist: " + str(nwdist)[0:4])
axes[1][0].scatter([], [], color = 'white', label = "CDE Loss: " + str(ncde)[0:4])
axes[1][0].legend(fontsize = 18, loc = 'upper right')
axes[1][0].set_ylim(0, 6)
axes[1][0].set_xlim(0, 2.5)
axes[1][0].set_title(' precise, \n not accurate', x=0.2, y=0.85)


axes[1][0].set_xlabel('Redshift')
axes[1][0].set_ylabel('$P(z|photometry)$')


# axes[1][1].plot(grid, base_dist.pdf(grid), color = 'k')
# axes[1][1].plot(np.ones(21), np.linspace(0, 20, 21), color = 'k', linestyle='dashed')
# axes[1][1].plot(grid, wide_dist.pdf(grid), color = 'red')
# axes[1][1].plot(1.2*np.ones(21), np.linspace(0, 20, 21), color = 'red', linestyle='dashed')

# axes[1][1].scatter([], [], color = 'white', label = "CvM: " + str(wcvm)[0:5])
# axes[1][1].scatter([], [], color = 'white', label = "KS: " + str(wks)[0:5])
# axes[1][1].scatter([], [], color = 'white', label = "RMSE: " + str(wrmse)[0:5])
# axes[1][1].scatter([], [], color = 'white', label = "KLD: " + str(wkld)[0:5])
# axes[1][1].scatter([], [], color = 'white', label = "W Dist: " + str(wwdist)[0:5])
# axes[1][1].scatter([], [], color = 'white', label = "CDF: " + str(wcdf)[0:5])
# axes[1][1].scatter([], [], color = 'white', label = "CDE Loss: " + str(wcde)[0:6])
# axes[1][1].legend(fontsize = 18, loc = 'upper right')
# axes[1][1].set_ylim(0, 11)
# axes[1][1].set_yticklabels([])
# axes[1][1].set_xlabel('Redshift')

#fig.supxlabel('Redshift')
#fig.supylabel('$P(z|photometry)$')
# Remove the bottom-right panel
fig.delaxes(axes[1][1])
fig.suptitle("Metric Responses to Precision and Accuracy", y=0.93)
fig.subplots_adjust(hspace = 0.05, wspace = 0.05)
fig.savefig("../plots/distribution_stats_plot.pdf", dpi=300, bbox_inches="tight")

#fig.tight_layout()

In [ ]:
# fig, axes = plt.subplots(nrows=2, ncols=3, figsize = (16, 8))

# grid = np.linspace(0, 3, 300)

# widecvm = sts.cramervonmises(base_dist.cdf(grid), wide_dist.cdf).statistic
# narrcvm = sts.cramervonmises(base_dist.cdf(grid), narrow_dist.cdf).statistic
# narr2cvm = sts.cramervonmises(base_dist.cdf(grid), narr2_dist.cdf).statistic

# axes[0][0].plot(grid, base_dist.pdf(grid), color='k')
# axes[0][0].plot(grid, wide_dist.pdf(grid), label = 'CvM = '+str(widecvm)[0:5])
# axes[0][0].plot(grid, narrow_dist.pdf(grid), label = 'CvM = '+str(narrcvm)[0:6])
# #axes[0][0].plot(grid, narr2_dist.pdf(grid), label = 'CvM = '+str(narr2cvm)[0:6])
# axes[0][0].legend()

# wideks = sts.kstest(base_dist.cdf(grid), wide_dist.cdf(grid)).statistic
# narrks = sts.kstest(base_dist.cdf(grid), narrow_dist.cdf(grid)).statistic
# narr2ks = sts.kstest(base_dist.cdf(grid), narr2_dist.cdf(grid)).statistic

# axes[0][1].plot(grid, base_dist.pdf(grid), color='k')
# axes[0][1].plot(grid, wide_dist.pdf(grid), label = 'KS = '+str(wideks)[0:5])
# axes[0][1].plot(grid, narrow_dist.pdf(grid), label = 'KS = '+str(narrks)[0:6])
# #axes[0][1].plot(grid, narr2_dist.pdf(grid), label = 'KS = '+str(narr2ks)[0:6])
# axes[0][1].legend()

# widermse = skl.mean_squared_error(base_dist.pdf(grid), wide_dist.cdf(grid))
# narrrmse = skl.mean_squared_error(base_dist.pdf(grid), narrow_dist.cdf(grid))

# axes[1][0].plot(grid, base_dist.pdf(grid), color='k')
# axes[1][0].plot(grid, wide_dist.pdf(grid), label = 'RMSE = '+str(widermse)[0:5])
# axes[1][0].plot(grid, narrow_dist.pdf(grid), label = 'RMSE = '+str(narrrmse)[0:6])
# axes[1][0].legend()

# basedata = base_dist.pdf(grid)
# widedata = wide_dist.pdf(grid)
# narrdata = narrow_dist.pdf(grid)

# basedata[basedata < 1e-300] = 1e-300
# widedata[widedata < 1e-300] = 1e-300
# narrdata[narrdata < 1e-300] = 1e-300

# widekld = sum(sps.kl_div(basedata, widedata))*3/300
# narrkld = sum(sps.kl_div(basedata, narrdata))*3/300

# print(widekld, narrkld)

# axes[1][1].plot(grid, base_dist.pdf(grid), color='k')
# axes[1][1].plot(grid, wide_dist.pdf(grid), label = 'KLD = '+str(widekld)[0:5])
# axes[1][1].plot(grid, narrow_dist.pdf(grid), label = 'KLD = '+str(narrkld)[0:6])
# axes[1][1].legend()

# axes[0][2].plot(grid, base_dist.pdf(grid), color='k')
# axes[0][2].plot(np.ones(21), np.linspace(0, 20, 21), color = 'k', linestyle='dashed')
# axes[0][2].plot(grid, wide_dist.pdf(grid), label = 'CDF = 0.345')
# axes[0][2].plot(grid, narrow_dist.pdf(grid), label = 'CDF = 7.620e-24')
# axes[0][2].legend()

# axes[1][2].plot(grid, base_dist.pdf(grid), color='k')
# axes[1][2].plot(np.ones(21), np.linspace(0, 20, 21), color = 'k', linestyle='dashed')
# axes[1][2].plot(grid, wide_dist.pdf(grid), label = 'CDE Loss = -0.909')
# axes[1][2].plot(grid, narrow_dist.pdf(grid), label = 'CDE Loss = 14.105')

# axes[1][2].legend()

# fig.suptitle('Effects of standard deviation on distribution-to-distribution and point-to-distribution statistics')

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# grid = np.linspace(0, 3, 300)

# widecvm = sts.cramervonmises(base_dist.cdf(grid), wide_dist.cdf).statistic
# narrcvm = sts.cramervonmises(base_dist.cdf(grid), narrow_dist.cdf).statistic
# narr2cvm = sts.cramervonmises(base_dist.cdf(grid), narr2_dist.cdf).statistic

# base1cvm = sts.cramervonmises(base_dist.cdf(grid), base1_dist.cdf).statistic
# base2cvm = sts.cramervonmises(base_dist.cdf(grid), base2_dist.cdf).statistic

# wideks = sts.kstest(base_dist.cdf(grid), wide_dist.cdf(grid)).statistic
# narrks = sts.kstest(base_dist.cdf(grid), narrow_dist.cdf(grid)).statistic
# narr2ks = sts.kstest(base_dist.cdf(grid), narr2_dist.cdf(grid)).statistic

# base1ks = sts.kstest(base_dist.cdf(grid), base1_dist.cdf(grid)).statistic
# base2ks = sts.kstest(base_dist.cdf(grid), base2_dist.cdf(grid)).statistic

# widermse = skl.mean_squared_error(base_dist.pdf(grid), wide_dist.cdf(grid))
# narrrmse = skl.mean_squared_error(base_dist.pdf(grid), narrow_dist.cdf(grid))

# base1rmse = skl.mean_squared_error(base_dist.pdf(grid), base1_dist.cdf(grid))
# base2rmse = skl.mean_squared_error(base_dist.pdf(grid), base2_dist.cdf(grid))

# basedata = base_dist.pdf(grid)
# widedata = wide_dist.pdf(grid)
# narrdata = narrow_dist.pdf(grid)

# base1data = base1_dist.pdf(grid)
# base2data = base2_dist.pdf(grid)

# basedata[basedata < 1e-300] = 1e-300
# widedata[widedata < 1e-300] = 1e-300
# narrdata[narrdata < 1e-300] = 1e-300

# base1data = base1_dist.pdf(grid)
# base2data = base2_dist.pdf(grid)

# widekld = sum(sps.kl_div(basedata, widedata))*3/300
# narrkld = sum(sps.kl_div(basedata, narrdata))*3/300

# base1kld = sum(sps.kl_div(basedata, base1data))*3/300
# base2kld = sum(sps.kl_div(basedata, base2data))*3/300


# plt.plot(grid, base_dist.pdf(grid), color='k')
# plt.plot(np.ones(21), np.linspace(0, 20, 21), color = 'k', linestyle='dashed')
# plt.plot(grid, narrow_dist.pdf(grid), color = 'orange',  label = 'CvM = '+str(narrcvm)[0:5])
# #plt.plot(grid, base2_dist.pdf(grid), color = 'dodgerblue', label = 'CvM = '+str(base2cvm)[0:6]) 

# # plt.plot([], [], color = 'orange',label = 'KS = '+str(wideks)[0:5] )
# # plt.plot([], [], color = 'dodgerblue', label = 'KS = '+str(narrks)[0:6])

# # plt.plot([], [], color = 'orange',  label = 'RMSE = '+str(widermse)[0:5])
# # plt.plot([], [], color = 'dodgerblue',  label = 'RMSE = '+str(narrrmse)[0:6])

# # plt.plot([], [], color = 'orange', label = 'KLD = '+str(widekld)[0:5])
# # plt.plot([], [], color = 'dodgerblue',  label = 'KLD = '+str(narrkld)[0:6])

# # plt.plot([], [], color = 'orange', label = 'CDF = 0.345')
# # plt.plot([], [], color = 'dodgerblue', label = 'CDF = 7.620e-24')

# # plt.plot([], [], color = 'orange', label = 'CDE Loss = -0.909')
# # plt.plot([], [], color = 'dodgerblue', label = 'CDE Loss = 14.105')

# plt.legend(prop = {'size': 15})

# plt.title('Distribution shape biases metrics differently')
# plt.xlabel('redshift z')
# plt.ylabel('probability density')